In [ ]:
"""This script collects public holiday data for various countries from July 2011 to December 2024 and saves it in a CSV file."""

from datetime import date
import holidays
import csv

def is_leap_year(year):
    """Determine whether a year is a leap year."""
    return year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)

# List of country codes
countries = [
    'US', 'GB', 'CA', 'AU', 'UA', 'RU', 'FR', 'DE', 'BR', 'CN', 'JP', 'PK',
    'KP', 'KR', 'IN', 'TW', 'NL', 'ES', 'SE', 'MX', 'IR', 'IL', 'SA', 'SY',
    'FI', 'IE', 'AT', 'NO', 'CH', 'IT', 'MY', 'EG', 'TR', 'PT', 'PS', 'AE'
]

# Initialize the list to store holiday data
H = []

# List of months
months = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']

# Prepare the header for the CSV file
header = ['Country']
for year in range(2011, 2025):  # Extend the range to 2024
    for month in months:
        # Skip months before July 2011
        if int(month) < 7 and year == 2011:
            continue
        header.append(f"{month}/{year}")

H.append(header)

# List to track countries with missing holiday data
missing = []

# Loop through each country
for country in countries:
    country_holidays = []
    for year in range(2011, 2025):  # Extend the range to 2024
        for month in months:
            # Skip months before July 2011
            if int(month) < 7 and year == 2011:
                continue

            counter = 0

            # Loop through each day of the month
            for day in range(1, 32):
                # Skip invalid dates
                if month == '02' and day > 28 and not is_leap_year(year):
                    continue
                if month == '02' and day > 29:
                    continue
                if day > 30 and (month in ['04', '06', '09', '11']):
                    continue

                try:
                    # Get the holidays for the country
                    c_holidays = holidays.country_holidays(country)
                except:
                    # If the country is not supported, add it to the missing list
                    if country not in missing:
                        missing.append(country)
                    continue

                # Check if the date is a holiday
                if date(year, int(month), day) in c_holidays:
                    counter += 1

            # Append the count of holidays for the month and year
            country_holidays.append(counter)

    # Append the holiday data for the country to the main list
    H.append([country] + country_holidays)
    print(f"Added holidays of {country}")

# Write the data to a CSV file
with open('PH_2011_2024.csv', "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(list(map(list, zip(*H))))  # Transpose the data for better readability

# Print any countries with missing holiday data
if missing:
    print("Holiday data not available for the following countries:", missing)

In [ ]:
from elsapy.elsclient import ElsClient
from elsapy.elsprofile import ElsAuthor, ElsAffil
from elsapy.elsdoc import FullDoc, AbsDoc
from elsapy.elssearch import ElsSearch
import json
import sys
import csv
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import time
import pandas as pd
from webdriver_manager.firefox import GeckoDriverManager

## Load configuration
con_file = open("config.json") #Place your Elsevier key in this file
config = json.load(con_file)
con_file.close()

## Initialise client
client = ElsClient(config['apikey'])

#harvester config (used to count the number of mentions)
options = Options()
options.add_argument('--headless')
driver = webdriver.Firefox(options=options, service=Service(GeckoDriverManager().install()))#executable_path = 'C:/Program Files/Mozilla Firefox/geckodriver')



# this list will be used in the query of Elsevier (attack list)
attack_list=['"Denial-of-Service Attack" OR {DDoS}',
'Phishing',
'Ransomware',
'Password Attack',
'"SQL Injection"',
'("Account Hijack*") OR ("Account Takeover")',
'Website Deface*',
'Trojan Attack', 
'Vulnerabilit* Attack exploit*', 
'"Zero-day" AND Attack',
'"Advanced Persistent Threat"',
'"cross-site scripting"',
'Malware OR TROJAN OR SPYWARE OR MALVERTIS* OR RANSOMWARE OR ("COMPUTER VIRUS") OR (WORM ATTACK) OR (KEYLOG* Attack) OR (KEYSTROKE LOG Attack) OR (MALICIOUS CODE) OR (MALICIOUS SOFTWARE) OR ADWARE OR ROOTKIT OR BOTNET OR (BACKDOOR ATTACK)',
'"Data Breach" OR "Data Leak*" OR "information leak*" OR "Data SPILL" OR "unintentional information disclos*"', 
'(Disinformation spread) OR (Misinformation spread) OR ("false information" spread) OR ("misleading information" spread) OR Deepfake',
'"Targeted Attack"',
'Adware',
'"Brute Force" Attack',
'Malvertis*',
'Backdoor Attack',
'Botnet',
'Cryptojack*',
'Worm Attack',
'Spyware',
'"Man-in-the-middle" OR {MITM}',
'"DNS Spoof*" OR "DNS cache poison*"',
'Pegasus Spyware',
'CoolWebSearch',
'Gator GAIN',
'"180search Assistant"',
'Transponder *vx2*',
'WannaCry AND (Ransomware OR Attack)',
'"Colonial Pipeline" AND (Ransomware OR Attack)',
'Cryptolocker OR Crypto-locker',
'Dropper Trojan',
'Wiper malware',
'Pharming Attack',
'"Insider Threat"',
'(Drive-by Download) OR (Drive-by Install)',
'Rootkit',
'Adversarial Attack',
'"Data Poisoning"',
'Deepfake',
'Deeplocker OR (Deep-locker) OR "Deep locker"',
'"Supply Chain" AND Attack',
'IoT Device Attack',
'(KEYLOG* Attack) OR (KEYSTROKE LOG Attack)',
'"DNS Tunnel*"',
'"Session Hijacking" OR "cookie hijacking" OR "cookie poisoning"',
'URL Attack',
'"Unknown Attack" AND (Security OR Cyber*)'
]


# this list will be used to count the number of mentions of each attack type by the harvester (attack keywords list)
mention_list=[('Denial-of-Service attack' , 'DDoS'),
              ('Phishing',),
              ('Ransomware',),
              ('Password',),
              ('SQL Injection','SQLI'),
              ('Account Hijack','Account Takeover'),
              ('Deface',),
              ('Trojan',),
              ('Vulnerability'),
              ('Zero-day',),
              ('Advanced Persistent Threat','APT Attack'),
              ('cross-site scripting',),
              ('Malware', 'TROJAN', 'SPYWARE', 'MALVERTIS',  'RANSOMWARE', 'VIRUS', 'WORM', 'KEYLOG', 'KEYSTROKE LOG', 'MALICIOUS CODE', 'MALICIOUS SOFTWARE', 'ADWARE', 'ROOTKIT', 'BOTNET', 'BACKDOOR'),
              ('Data Breach', 'Data Leak', 'information leak', 'Data SPILL', 'unintentional information disclos'),
              ('Disinformation', 'Misinformation', 'false information', 'misleading information', 'Deepfake'),
              ('Targeted Attack',),
              ('Adware',),
              ('Brute Force',),
              ('Malvertis',),
              ('Backdoor',),
              ('Botnet',),
              ('Cryptojack',),
              ('Worm',),
              ('Spyware',),
              ('Man-in-the-middle', 'MITM'), 
              ('DNS Spoof', 'DNS cache poison'),
              ('Pegasus',),
              ('CoolWebSearch',),
              ('Gator Spyware','GAIN Spyware','GAIN) Spyware','Gator (GAIN)','Gator/GAIN','Gator GAIN'),
              ('180search Assistant',),
              ('Transponder vx2','Transponder Spyware','vx2 Spyware','vx2) Spyware','Transponder/vx2', 'Transponder (vx2)'),
              ('WannaCry',),
              ('Colonial Pipeline',),
              ('Cryptolocker', 'Crypto-locker'),
              ('Dropper',),
              ('Wiper',),
              ('Pharming',),
              ('Insider Threat',),
              ('Drive-by',),
              ('Rootkit',),
              ('Adversarial Attack',),
              ('Data Poisoning',),
              ('Deepfake',),
              ('Deeplocker','Deep locker'),
              ('Supply Chain',),
              ('IoT Device','IoT Attack','Attack on IoT','Attack the IoT'),
              ('KEYLOGGER','KEYSTROKE LOG'),
              ('DNS Tunnel',),
              ('Session Hijack','Hijack Session','Hijack the Session', 'cookie hijack', 'cookie poison'),
              ('URL manipul' , 'URL tamper' , 'URL interpret' , 'URL Attack'),
              ('Unknown Attack',)
             ]                       
              



years_list=list(range(2011,2025));
month_list=['January','February','March','April','May','June','July','August','September','October','November','December']

empty='Result set was empty'

search_results_list=[]

#Part 1 - collect the URLs of all relevant documents (for each month and each attack type)
#The search is within the title, abstract, and keywords
for year in years_list:
    print ("searching in year:", str(year))
    
    for month in month_list:

        #Our dataset starts from July 2011
        if year==2011 and month in month_list[:6]:
            continue;

        
        print ("searching in year/month:", str(year),"/",month)
        slsm=[]
        for index,attack in enumerate(attack_list):
            slm=[]
            print("searching for ",attack)
            doc_srch = ElsSearch("TITLE("+attack+") OR ABS("+attack+") OR KEY("+attack+") AND PUBDATETXT("+month+" "+str(year)+")",'scopus')
            doc_srch.execute(client, get_all = True)
            if(empty in str(doc_srch.results)):
                #print(doc_srch.results)
                slsm.append(slm);
                continue
            else:
                #print("got" , len(doc_srch.results))
                for col in doc_srch.results:
                    slm.append(col)
                slsm.append(slm);
        search_results_list.append(slsm)
        
  




#Part 2 - Harvesting and counting the number of mentions of each attack type during each month
#The counting is within the title, abstract, and keywords
print ("--------------------------Harvesting Time ---------------------------")
data=[]
for index,row in enumerate(search_results_list):
    print("Out of", len(search_results_list)," Harvesting for column ",index)
    attack_map = {}
    for mention in mention_list:
        attack_map[mention[0]]=0;
    for j, col in enumerate(row):
        for k, result in enumerate(col):
            url=result['link'][2]['@href']
            print("URL:",url)
            try:
                driver.get(url)
            except:
                time.sleep(20)
                driver.get(url)
            time.sleep(1)
            delay = 20 # seconds
            try:
                myElem = WebDriverWait(driver, delay).until(EC.presence_of_element_located((By.CLASS_NAME, 'Highlight-module__akO5D')))
            except TimeoutException:
                print ("Loading took too much time!")
            #click index keywords
            keyword_elements =  driver.find_elements(By.TAG_NAME,"button")
            keyword_element  = None
            while(True):
                failed=False
                for element in keyword_elements :
                    try:
                        if('Indexed keywords' in element.text):
                            keyword_element=element
                            print("keyword element found!")
                            break;
                    except:
                        print("button issue")
                        keyword_element=None
                        failed=True
                        time.sleep(1)
                        break;
                if failed==False:
                     break
                else:
                     keyword_elements =  driver.find_elements(By.TAG_NAME,"button")

            if(keyword_element is not None) :       
                keyword_element.click()
                time.sleep(1)
            mention = mention_list[j]
            
            #counting
            while (True):
                failed=False
                totalQ=0;
                text_elements =  driver.find_elements(By.CLASS_NAME,"Highlight-module__akO5D");
                for element in text_elements:
                    try:
                        element_text=element.text
                    except:
                        failed=True;
                        break;
                    count = sum(element_text.replace("-"," ").upper().count(x.upper().replace("-"," ")) for x in mention)
                    print("count:",count)
                    totalQ+=count
                if(not failed):
                    break;
            attack_map[mention[0]]+=totalQ
            print("for attack",mention,"sum is now:",  attack_map[mention[0]])
    data_list=[]
    for key in attack_map:
        print(key,": ",attack_map[key])
        data_list.append(attack_map[key])
    data.append(data_list)  
    
    #save data to file
    file_object = open('Attacks_NoM.txt', 'a')
    for ii,item in enumerate(data_list):
        file_object.write(str(item))
        if ii<len(data_list)-1:
            file_object.write("\t")
    file_object.write("\n")
    file_object.close()

1. Place the Elsevier API key in config.json and then run the script

In [ ]:
from elsapy.elsclient import ElsClient
from elsapy.elsprofile import ElsAuthor, ElsAffil
from elsapy.elsdoc import FullDoc, AbsDoc
from elsapy.elssearch import ElsSearch
import json
import sys
import csv
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import time
import pandas as pd
from webdriver_manager.firefox import GeckoDriverManager

 
## Load configuration
con_file = open("config.json")#Place your Elsevier key in this file
config = json.load(con_file)
con_file.close()

## Initialise client
client = ElsClient(config['apikey'])

#harvester config (used to count the number of mentions)
options = Options()
options.add_argument('--headless')
driver = webdriver.Firefox(options=options, service=Service(GeckoDriverManager().install()))#executable_path = 'C:/Program Files/Mozilla Firefox/geckodriver')



# this list will be used in the query of Elsevier (PT list)
solution_list=['BLOCKCHAIN',
'"ACCESS CONTROL"',
'ENCRYPTION',
'"SUPPLY CHAIN" AND "RISK MANAGEMENT"',
'"IDENTITY MANAGEMENT"',
'"Double Patterning Lithography"',
'"MACHINE LEARNING"',
'"ANOMALY DETECTION"', 
'CRYPTOGRAPHY',
'"PENETRATION TESTING"',
'"intrusion detection" OR "intrusion prevention"',
'"STATIC ANALYSIS"',
'"DYNAMIC ANALYSIS"',
'"MULTI FACTOR AUTHENTICATION" OR "MULTIFACTOR AUTHENTICATION" OR "MULTI-FACTOR AUTHENTICATION"', 
'"LEAST PRIVILEGE"',
'"SESSION MANAGEMENT"',
'CAPTCHA',
'BLACKLISTING',
'"RATE LIMITING"',
'"GRAPHICAL MODEL*"',
'HONEYPOT',
'"SOFTWARE DEFINED NETWORK"',
'"GAME THEORY"',
'"GRAPH MACHINE LEARNING" OR "GRAPH-BASED MACHINE LEARNING"',
'"IP WHITELIST*"',
'"TRAFFIC SHAPING"',
'"PACKET FILTERING"',
'BLACKHOLING OR "BLACK HOLING" OR ("Black hol*" AND network)',
'"RANK CORRELATION"',
'D3NS AND DNS',
'"SESSION ID RANDOMIZATION" OR "SESSION ID RANDOMISATION"',
'"STRONG AUTHENTICATION" OR Kerberos',
'"SECURE SOCKETS LAYER" OR "Transport Layer Security"',
'{HTTPS}',
'"CONTINUOUS AUTHENTICATION"',
'"Identity-Based Encryption" OR "Identity Based Encryption"',
'"DATA SANITIZATION" OR "DATA SANITISATION"',
'"OUTLIER DETECTION"',
'"DATA PROVENANCE"',
'"ADVERSARIAL TRAINING" OR "adversarial learning"',
'"TRUSTWORTHY AI"',
'"DEEP PROBABILISTIC MODEL*"',
'"Bayesian Network*"',
'"TROJAN ISOLATION"',
'"HARDWARE SANDBOXING"',
'"BEHAVIOR BASED DETECTION" OR "BEHAVIOR-BASED DETECTION" OR "BEHAVIOUR BASED DETECTION" OR "BEHAVIOUR-BASED DETECTION"',
'"FORMAL VERIFICATION"',
'"SPLIT MANUFACTURING"',
'"VULNERABILITY MANAGEMENT"',
'"FILE INTEGRITY" AND MONITOR*',
'{VPN}',
'"CSS MATCHING"',
'"URI MATCHING"',
'"PRIVACY PRESERVING"',
'"SECURE BOOT"',
'"MERKLE SIGNATURE"',
'"LIVENESS DETECTION"',
'"AUDIO DEEPFAKE DETECTION"',
'"3D FACE RECONSTRUCTION"',
'BIOMETRICS',
'"DIGITAL WATERMARK"',
'"APPLICATION WHITELISTING"',
'"DATA BACKUP"',
'"HIDDEN MARKOV MODEL"',
'"PATCH MANAGEMENT"',
'"DATA AUGMENTATION"',
'"DIMENSIONALITY REDUCTION"',
'"DEFENSIVE DISTILLATION"',
'"GRADIENT MASKING"',
'{RRAM}',
'"SPATIAL SMOOTHING"',
'"NOISE INJECTION"',
'"TAINT ANALYSIS"',
'"NETWORK SEGMENTATION"',
'"USER BEHAVIOR ANALYTICS" OR "USER BEHAVIOUR ANALYTICS" OR ("USER BEHAVIOR" AND ANALYTICS) OR ("USER BEHAVIOUR" AND ANALYTICS)',
'"DECEPTION TECHNOLOGY"',
'"RISK ASSESSMENT"',
'"LOG CORRELATION"',
'"DYNAMIC RESOURCE MANAGEMENT"',
'"SANDBOXING"',
'"DARKNET MONITORING" OR "MONITOR* DARKNET" OR "DARKNET MONITOR*"',
'"VIRTUAL KEYBOARD*"',
'"CODE SIGN*"',
'"FILE SIGNATURE"',
'"PUBLIC KEY INFRASTRUCTURE"',
'"VOICEPRINT AUTHENTICATION"',
'"MUTUAL AUTHENTICATION"',
'"PASSWORD SALT*"',
'"ONE TIME PASSWORD"',
'"DYNAMIC BINARY INSTRUMENTATION"',
'"ORTHOGONAL OBFUSCATION"',
'"PASSWORD HASH*"',
'{DNSSEC}',
'"CERTIFICATE PINNING"',
'"SECURE SIMPLE PAIRING"',
'"VULNERABILITY ASSESSMENT"',
'{SIEM}',
'"STANDARDIZED COMMUNICATION" OR "STANDARDISED COMMUNICATION"',
'"Control Flow Integrity"',
'"VULNERABILITY SCAN*"',
'"PASSWORD STRENGTH METER*"',
'"PASSWORD MANAGEMENT"',
'"PASSWORD POLICY" OR "PASSWORD POLICI*"',
'"GRAPHICAL AUTHENTICATION"',
'"DATA LOSS PREVENTION"',
'"DATA LEAKAGE DETECTION" OR "DATA LEAKAGE PREVENTION"',
'"ACTIVITY MONITORING"',
'"MOVING TARGET DEFEN*"',
'"KEYSTROKE DYNAMICS"',
'"ATTACK TREE"',
'"Automatic Violation Prevention"',
'"DISTRIBUTED LEDGER*"',
'"SOURCE IDENTIFICATION"',
'"IMAGE RECOGNITION"',
'{MAUVE}',
'"HYPERGAME"',
'"PREBUNKING"',
'"natural language processing" OR "language model*"']


# this list will be used to count the number of mentions of each PT by the harvester (PT keywords list)
mention_list=[('BLOCKCHAIN',),
              ('ACCESS CONTROL',),
              ('ENCRYPTION',),
              ('RISK MANAGEMENT',),
              ('IDENTITY MANAGEMENT',),
              ('Double Patterning Lithography',),
              ('MACHINE LEARNING',),
              ('ANOMALY DETECTION',),
              ('CRYPTOGRAPHY',),
              ('PENETRATION TESTING',),
              ('intrusion detection','intrusion prevention'),
              ('STATIC ANALYSIS',),
              ('DYNAMIC ANALYSIS',),
              ('MULTI FACTOR AUTHENTICATION','MULTIFACTOR AUTHENTICATION','MULTI-FACTOR AUTHENTICATION'),
              ('LEAST PRIVILEGE',),
              ('SESSION MANAGEMENT',),
              ('CAPTCHA',),
              ('BLACKLISTING',),
              ('RATE LIMITING',),
              ('GRAPHICAL MODEL',),
              ('HONEYPOT',),
              ('SOFTWARE DEFINED NETWORK',),
              ('GAME THEORY',),
              ('GRAPH MACHINE LEARNING','GRAPH-BASED MACHINE LEARNING','GRAPH BASED MACHINE LEARNING'),
              ('WHITELIST',),
              ('TRAFFIC SHAP',),
              ('PACKET FILTER',),
              ('BLACKHOL','BLACK HOL'),
              ('RANK CORRELATION',),
              ('D3NS',),
              ('ID RANDOMIZATION','ID RANDOMISATION'),
              ('STRONG AUTHENTICATION',),
              ('SECURE SOCKETS LAYER','Transport Layer Security','SSL','TLS'),
              ('HTTPS',),             
              ('CONTINUOUS AUTHENTICATION',),
              ('Identity-Based Encryption','Identity Based Encryption'), 
              ('DATA SANITIZATION','DATA SANITISATION'),
              ('OUTLIER DETECTION',),
              ('DATA PROVENANCE',),
              ('ADVERSARIAL TRAINING','adversarial learning'),
              ('TRUSTWORTHY AI',),
              ('DEEP PROBABILISTIC MODEL',),
              ('Bayesian Network',),
              ('TROJAN ISOLATION',),
              ('HARDWARE SANDBOXING',),
              ('BEHAVIOR BASED DETECTION','BEHAVIOR-BASED DETECTION','BEHAVIOUR BASED DETECTION','BEHAVIOUR-BASED DETECTION'),
              ('FORMAL VERIFICATION',),
              ('SPLIT MANUFACTURING',),
              ('VULNERABILITY MANAGEMENT',),
              ('FILE INTEGRITY',),
              ('VPN',),
              ('CSS MATCH',),
              ('URI MATCH',),
              ('PRIVACY PRESERV',),
              ('SECURE BOOT',),              
              ('MERKLE SIGNATURE',),
              ('LIVENESS DETECTION',),              
              ('AUDIO DEEPFAKE DETECTION',),
              ('FACE RECONSTRUCTION',),
              ('BIOMETRIC',), 
              ('DIGITAL WATERMARK',),
              ('APPLICATION WHITELISTING',),
              ('Data BACKUP',),
              ('HIDDEN MARKOV',),
              ('PATCH MANAGEMENT',),
              ('DATA AUGMENTATION',),
              ('DIMENSIONALITY REDUCTION',),
              ('DEFENSIVE DISTILLATION',),
              ('GRADIENT MASK',),
              ('RRAM',),
              ('SPATIAL SMOOTHING',),
              ('NOISE INJECTION',),
              ('TAINT ANALYSIS',),
              ('NETWORK SEGMENTATION',),
              ('USER BEHAVIOR ANALYTICS','USER BEHAVIOUR ANALYTICS'),
              ('DECEPTION TECHNOLOGY',),
              ('RISK ASSESSMENT',),
              ('LOG CORRELATION',),
              ('DYNAMIC RESOURCE MANAGEMENT',),
              ('SANDBOXING',),
              ('DARKNET MONITORING',),
              ('VIRTUAL KEYBOARD',),
              ('CODE SIGN',),
              ('FILE SIGNATURE',),
              ('PUBLIC KEY INFRASTRUCTURE',),
              ('VOICEPRINT AUTHENTICATION',),
              ('MUTUAL AUTHENTICATION',),
              ('PASSWORD SALT',),
              ('ONE TIME PASSWORD',),
              ('DYNAMIC BINARY INSTRUMENTATION',),
              ('ORTHOGONAL OBFUSCATION',),
              ('PASSWORD HASH',),
              ('DNSSEC',),
              ('CERTIFICATE PINNING',),
              ('SECURE SIMPLE PAIRING',),
              ('VULNERABILITY ASSESSMENT',),
              ('SIEM',),
              ('STANDARDIZED COMMUNICATION','STANDARDISED COMMUNICATION'),
              ('Control Flow Integrity',),
              ('VULNERABILITY SCAN',),
              ('PASSWORD STRENGTH METER',),
              ('PASSWORD MANAGEMENT',),
              ('PASSWORD POLICY','PASSWORD POLICIES'),
              ('GRAPHICAL AUTHENTICATION',),
              ('DATA LOSS PREVENTION',),
              ('DATA LEAKAGE DETECTION','DATA LEAKAGE PREVENTION'),
              ('ACTIVITY MONITORING',),
              ('MOVING TARGET DEFEN',),
              ('KEYSTROKE DYNAMICS',),
              ('ATTACK TREE',),
              ('Automatic Violation Prevention',),
              ('DISTRIBUTED LEDGER',),
              ('SOURCE IDENTIFICATION',),
              ('IMAGE RECOGNITION',),
              ('MAUVE',),
              ('HYPERGAME',),
              ('PREBUNKING',),
              ('NATURAL LANGUAGE PROCESS','language model','NLP','LLM')]             
              


years_list=list(range(2011,2025));
month_list=['January','February','March','April','May','June','July','August','September','October','November','December']

empty='Result set was empty'

search_results_list=[]

#Part 1 - collect the URLs of all relevant documents (for each month and each PT)
#The search is within the title, abstract, and keywords
for year in years_list:
    print ("searching in year:", str(year))
    
    for month in month_list:

        #Our dataset starts from July 2011
        if year==2011 and month in month_list[:6]:
            continue;

        
        print ("searching in year/month:", str(year),"/",month)
        slsm=[]
        for index,solution in enumerate(solution_list):
            slm=[]
            print("searching for ",solution)
            doc_srch = ElsSearch("(TITLE("+solution+") OR ABS("+solution+") OR KEY("+solution+")) AND (TITLE(*secur*) OR ABS(*secur*) OR KEY(*secur*)) AND (PUBDATETXT("+month+" "+str(year)+"))",'scopus')
            doc_srch.execute(client, get_all = True)
            if(empty in str(doc_srch.results)):
                #print(doc_srch.results)
                slsm.append(slm);
                continue
            else:
                #print("got" , len(doc_srch.results))
                for col in doc_srch.results:
                    slm.append(col)
                slsm.append(slm);
        search_results_list.append(slsm)
        

#Part 2 - Harvesting and counting the number of mentions of each PT during each month
#The counting is within the title, abstract, and keywords
print ("--------------------------Harvesting Time ---------------------------")
data=[]
for index,row in enumerate(search_results_list):
    print("Out of", len(search_results_list)," Harvesting for column ",index)
    attack_map = {}
    for mention in mention_list:
        attack_map[mention[0]]=0;
    for j, col in enumerate(row):
        for k, result in enumerate(col):
            url=result['link'][2]['@href']
            print("URL:",url)
            try:
                driver.get(url)
            except:
                time.sleep(20)
                driver.get(url)
            time.sleep(1)
            delay = 20 # seconds
            try:
                myElem = WebDriverWait(driver, delay).until(EC.presence_of_element_located((By.CLASS_NAME, 'Highlight-module__akO5D')))
            except TimeoutException:
                print ("Loading took too much time!")
            #click index keywords
            keyword_elements =  driver.find_elements(By.TAG_NAME,"button")
            keyword_element  = None
            while(True):
                failed=False
                for element in keyword_elements :
                    try:
                        if('Indexed keywords' in element.text):
                            keyword_element=element
                            print("keyword element found!")
                            break;
                    except:
                        print("button issue")
                        keyword_element=None
                        failed=True
                        time.sleep(1)
                        break;
                if failed==False:
                     break
                else:
                     keyword_elements =  driver.find_elements(By.TAG_NAME,"button")

            if(keyword_element is not None) :       
                keyword_element.click()
                time.sleep(1)
            mention = mention_list[j]
            
            #counting
            while (True):
                failed=False
                totalQ=0;
                text_elements =  driver.find_elements(By.CLASS_NAME,"Highlight-module__akO5D");
                for element in text_elements:
                    try:
                        element_text=element.text
                    except:
                        failed=True;
                        break;
                    count = sum(element_text.replace("-"," ").upper().count(x.upper().replace("-"," ")) for x in mention)
                    print("count:",count)
                    totalQ+=count
                if(not failed):
                    break;
            attack_map[mention[0]]+=totalQ
            print("for solution",mention,"sum is now:",  attack_map[mention[0]])
    data_list=[]
    for key in attack_map:
        print(key,": ",attack_map[key])
        data_list.append(attack_map[key])
    data.append(data_list)  
    
    #save data to file
    file_object = open('PTs_NoM.txt', 'a')
    for ii,item in enumerate(data_list):
        file_object.write(str(item))
        if ii<len(data_list)-1:
            file_object.write("\t")
    file_object.write("\n")
    file_object.close()

1. Place the Elsevier API key in config.json and then run the script

In [ ]:
# RSS security feeds

"""
This script collects security data from over 150 RSS feeds, processes it to count daily occurrences of security incidents and mentions of solutions by type and source, and aggregates to monthly counts.

"""

import asyncio
import aiohttp
import feedparser
import csv
from datetime import datetime, timedelta
import re
from collections import defaultdict
import pandas as pd
import numpy as np
from functools import lru_cache
import time
from typing import List, Dict, Tuple
import logging

# Suppress feedparser warnings for cleaner output
logging.getLogger('feedparser').setLevel(logging.ERROR)

# Configuration
FEEDS = [
        # Security Blogs/News
    "https://www.imperialviolet.org/iv-rss.xml",
    "https://research.kudelskisecurity.com/feed/",
    "http://blog.uncommonsensesecurity.com/feeds/posts/default",
    "http://www.darkreading.com/rss/all.xml",
    "http://feeds.feedburner.com/schneier/fulltext",
    "http://blog.trailofbits.com/feed/",
    "https://www.malwaretech.com/feed/",
    "https://krebsonsecurity.com/feed/",
    "https://www.bleepingcomputer.com/feed/",
    "https://threatpost.com/feed/",
    "https://www.csoonline.com/index.rss",
    "https://www.infosecurity-magazine.com/rss/news/",
    "https://www.zdnet.com/topic/security/rss.xml",
    "https://www.welivesecurity.com/feed/",
    "https://www.tripwire.com/state-of-security/feed/",
    "https://www.securityweek.com/rss.xml",
    "https://www.helpnetsecurity.com/feed/",
    "https://www.scmagazine.com/home/security-news/feed/",
    "https://www.itsecurityguru.org/feed/",
    "https://www.cnet.com/feeds/security/",
    "https://www.techrepublic.com/rssfeeds/topic/security/",
    "https://www.thehackernews.com/feeds/posts/default",
    "https://www.infosecurity-magazine.com/rss/blogs/",
    "https://www.cyberscoop.com/feed/",
    "https://www.securityaffairs.co/wordpress/feed",
    "https://www.cyberdefensemagazine.com/feed/",
    "https://www.infosecurity-magazine.com/rss/reviews/",
    "https://www.infosecurity-magazine.com/rss/opinions/",
    "https://www.infosecurity-magazine.com/rss/features/",
    "https://www.infosecurity-magazine.com/rss/analysis/",
    "https://www.infosecurity-magazine.com/rss/interviews/",
    "https://www.infosecurity-magazine.com/rss/events/",
    "https://www.infosecurity-magazine.com/rss/webinars/",
    "https://www.infosecurity-magazine.com/rss/whitepapers/",
    "https://www.infosecurity-magazine.com/rss/infographics/",
    "https://www.ncsc.gov.uk/api/1/services/v1/all-rss-feed.xml",
    "https://www.us-cert.gov/ncas/alerts.xml",
    "https://www.schneier.com/blog/atom.xml",
    "https://www.dspace.com/rsssecurity.xml",
    "https://www.f-secure.com/en/web/labs_global/rss",
    "https://www.grahamcluley.com/feed/",
    "https://www.malwarebytes.com/blog/feed/",
    "https://seclists.org/rss/basics.rss",
    "https://seclists.org/rss/fulldisclosure.rss",
    "https://seclists.org/rss/pen-test.rss",
    "https://seclists.org/rss/bugtraq.rss",
    "https://seclists.org/rss/webappsec.rss",
    "https://seclists.org/rss/oss-sec.rss",
    "https://seclists.org/rss/crypto.rss",
    "https://seclists.org/rss/packetstorm.rss",
    "https://seclists.org/rss/zeroday.rss",

    # Security Advisories & Alerts
    "https://www.ncsc.gov.uk/api/1/services/v1/rss-feed.xml",
    "https://msrc.microsoft.com/blog/rss/",
    "https://www.zerodayinitiative.com/rss/",
    "https://cvefeed.io/rssfeed",
    "https://www.cisa.gov/cybersecurity-advisories.xml",
    "https://www.nist.gov/rss-feeds.xml",
    "https://www.bsi.bund.de/SiteGlobals/Forms/RSSFeed/RSSNewsfeed_Formular.xml?nn=784524",

    # Security Blogs & News
    "https://news.sophos.com/en-us/feed/",
    "https://aws.amazon.com/blogs/security/feed/",
    "https://blogs.cisco.com/security/feed",
    "https://feeds.trendmicro.com/TrendMicroSecurityBlog",
    "https://blog.qualys.com/feed/",
    "https://gbhackers.com/feed/",
    "https://www.hackingarticles.in/feed/",
    "https://www.sentinelone.com/feed/",
    "https://www.troyhunt.com/rss/",
    "https://securelist.com/feed/",
    "https://cloudsecurityalliance.org/feed/",
    "https://www.bankinfosecurity.com/rss-feeds",
    "https://www.blackhillsinfosec.com/feed/",
    "https://www.govinfosecurity.com/rss-feeds",
    "https://www.benthamsgaze.org/feed/",
    "https://bishopfox.com/feeds/blog.rss",
    "https://www.veracode.com/blog/feed",
    "https://marcoramilli.com/feed/",
    "https://robert.penz.name/feed/",
    "https://feeds.feedburner.com/TheHackerNews",
    "https://www.securityweek.com/feed/",
    "https://www.tripwire.com/state-of-security/rss",
    "https://www.secureworks.com/blog/rss",
    "https://securityledger.com/feed/",
    "https://blog.skullsecurity.org/feed/",
    "https://holisticinfosec.blogspot.com/feeds/posts/default",
    "https://taosecurity.blogspot.com/feeds/posts/default",
    "https://blog.rootshell.be/feed/",
    "https://adamlevin.com/feed/",
    "https://johnmasserini.com/feed/",
    "https://alstonprivacy.com/feed/",
    "https://defensivesecurity.org/feed/",
    "https://paloaltonetworks.com/blog/feed/",
    "https://security.googleblog.com/feeds/posts/default",
    "https://blog.malwarebytes.com/feed/",
    "https://www.avast.com/blog/rss.xml",
    "https://blog.sucuri.net/feed/",
    "https://research.checkpoint.com/feed/",
    "https://www.crowdstrike.com/blog/feed/",
    "https://blog.f-secure.com/feed/",
    "https://blogs.jpcert.or.jp/en/atom.xml",
    "https://isc.sans.edu/rssfeed.xml",
    "https://www.cert.ssi.gouv.fr/feed/",
    "https://www.trendmicro.com/rss/all.xml",
    "https://www.darktrace.com/en/rss-feed.xml",
    "https://www.elastic.co/security-labs/feed.xml",
    "https://www.okta.com/security-blog/rss/",
    "https://blog.cloudflare.com/tag/security/rss/",
    "https://www.microsoft.com/security/blog/feed/",
    "https://www.us-cert.gov/ncas/all.xml",
    "https://www.talosintelligence.com/feed",
    "https://www.fortinet.com/blog.rss",
    "https://www.mcafee.com/blogs/feed/",
    "https://community.rsa.com/blogs/feed",
    "https://www.eset.com/int/rss/blog/",
    "https://www.ibm.com/blogs/security/feed/",
    "https://www.tenable.com/blog/feed",
    "https://blog.radware.com/feed/",
    "https://www.trustwave.com/en-us/rss/blogs/",
    "https://www.proofpoint.com/us/rss/blog.xml",
    "https://www.carbonblack.com/blog/feed/",
    "https://www.splunk.com/en_us/blog/security/_jcr_content.feed",
    "https://www.fireeye.com/blog/feed",
    "https://www.cybereason.com/blog/rss.xml",

    # Security Forums & Community
    "https://www.reddit.com/r/netsec/.rss",
    "https://security.stackexchange.com/feeds/week",
    "https://www.reddit.com/r/cybersecurity/.rss",
    "https://www.reddit.com/r/AskNetsec/.rss",
    "https://www.reddit.com/r/ReverseEngineering/.rss",
    "https://www.reddit.com/r/Malware/.rss",

    # Archival & Mailing List Feeds
    "https://catless.ncl.ac.uk/Risks/rss",
    "https://web.archive.org/web/20220101000000*/http://www.phrack.org/rss.xml",
    "https://marc.info/?rss",
    "https://allinfosecnews.com/feed/",
    "https://github.com/foorilla/allinfosecnews_sources/blob/main/sources.json",
    "https://github.com/thehappydinoa/awesome-threat-intel-rss",
    "https://www.cyberseo.net/public-rss-feed-catalogue/Information%20Security/",
    "https://archive.cert.org/rss/security-updates.xml",
    "https://archives.neohapsis.com/rss/",
    "https://lists.openwall.net/rss/all.xml",
    "https://www.packetstormsecurity.com/feeds/news/",
    "https://exploit-db.com/rss.xml",
    "https://seclists.org/rss/oss-security.rss",
    "https://vulners.com/rss",
    "https://www.exploit-db.com/rss/",
    "https://www.securityfocus.com/rss/vulnerabilities.xml",
    "https://www.nvd.nist.gov/feeds/xml/cve/misc/nvd-rss.xml",
    "https://feeds.feedburner.com/SecurityFocus",
    "https://archive.cert.europa.eu/rss/advisories.xml",
    "http://www.reddit.com/r/netsec/.rss",
    "http://security.stackexchange.com/feeds/week"
]

SECURITY_CATEGORIES = [
    'DDoS', 
    'Phishing', 
    'Ransomware', 
    'Vulnerability', 
    'Zero-day',
    'APT', 
    'Malware', 
    'Data_Breach', 
    'Account_Hijacking', 
    'Defacement',
    'Spyware', 
    'Unknown', 
    'Others',
    'Trojan',
    'Password_Attack',
    'SQL_Injection',
    'XSS',
    'Disinformation',
    'Targeted_Attack',
    'Adware',
    'Brute_Force',
    'Malvertising',
    'Backdoor',
    'Botnet',
    'Cryptojacking',
    'Worm',
    'MITM',
    'DNS_Spoofing',
    'Insider_Threat',
    'Drive-by',
    'Rootkit',
    'Adversarial_Attack',
    'Data_Poisoning',
    'Deepfake',
    'Deeplocker',
    'Supply_Chain',
    'IoT_Attack',
    'DNS_Tunneling',
    'Session_Hijacking',
    'URL_Attack',
    'Unknown_Attack'
]

SOLUTIONS_LIST = [
    'BLOCKCHAIN',
    'ACCESS CONTROL',
    'ENCRYPTION',
    'SUPPLY CHAIN RISK MANAGEMENT',
    'IDENTITY MANAGEMENT',
    'Double Patterning Lithography',
    'MACHINE LEARNING',
    'ANOMALY DETECTION', 
    'CRYPTOGRAPHY',
    'PENETRATION TESTING',
    'INTRUSION DETECTION PREVENTION',
    'STATIC ANALYSIS',
    'DYNAMIC ANALYSIS',
    'MULTI FACTOR AUTHENTICATION', 
    'LEAST PRIVILEGE',
    'SESSION MANAGEMENT',
    'CAPTCHA',
    'BLACKLISTING',
    'RATE LIMITING',
    'GRAPHICAL MODEL',
    'HONEYPOT',
    'SOFTWARE DEFINED NETWORK',
    'GAME THEORY',
    'GRAPH MACHINE LEARNING',
    'IP WHITELIST',
    'TRAFFIC SHAPING',
    'PACKET FILTERING',
    'BLACKHOLING',
    'RANK CORRELATION',
    'D3NS DNS',
    'SESSION ID RANDOMIZATION',
    'STRONG AUTHENTICATION',
    'SECURE SOCKETS LAYER',
    'HTTPS',
    'CONTINUOUS AUTHENTICATION',
    'Identity-Based Encryption',
    'DATA SANITIZATION',
    'OUTLIER DETECTION',
    'DATA PROVENANCE',
    'ADVERSARIAL TRAINING',
    'TRUSTWORTHY AI',
    'DEEP PROBABILISTIC MODEL',
    'Bayesian Network',
    'TROJAN ISOLATION',
    'HARDWARE SANDBOXING',
    'BEHAVIOR BASED DETECTION',
    'FORMAL VERIFICATION',
    'SPLIT MANUFACTURING',
    'VULNERABILITY MANAGEMENT',
    'FILE INTEGRITY MONITORING',
    'VPN',
    'CSS MATCHING',
    'URI MATCHING',
    'PRIVACY PRESERVING',
    'SECURE BOOT',
    'MERKLE SIGNATURE',
    'LIVENESS DETECTION',
    'AUDIO DEEPFAKE DETECTION',
    '3D FACE RECONSTRUCTION',
    'BIOMETRICS',
    'DIGITAL WATERMARK',
    'APPLICATION WHITELISTING',
    'DATA BACKUP',
    'HIDDEN MARKOV MODEL',
    'PATCH MANAGEMENT',
    'DATA AUGMENTATION',
    'DIMENSIONALITY REDUCTION',
    'DEFENSIVE DISTILLATION',
    'GRADIENT MASKING',
    'RRAM',
    'SPATIAL SMOOTHING',
    'NOISE INJECTION',
    'TAINT ANALYSIS',
    'NETWORK SEGMENTATION',
    'USER BEHAVIOR ANALYTICS',
    'DECEPTION TECHNOLOGY',
    'RISK ASSESSMENT',
    'LOG CORRELATION',
    'DYNAMIC RESOURCE MANAGEMENT',
    'SANDBOXING',
    'DARKNET MONITORING',
    'VIRTUAL KEYBOARD',
    'CODE SIGN',
    'FILE SIGNATURE',
    'PUBLIC KEY INFRASTRUCTURE',
    'VOICEPRINT AUTHENTICATION',
    'MUTUAL AUTHENTICATION',
    'PASSWORD SALT',
    'ONE TIME PASSWORD',
    'DYNAMIC BINARY INSTRUMENTATION',
    'ORTHOGONAL OBFUSCATION',
    'PASSWORD HASH',
    'DNSSEC',
    'CERTIFICATE PINNING',
    'SECURE SIMPLE PAIRING',
    'VULNERABILITY ASSESSMENT',
    'SIEM',
    'STANDARDIZED COMMUNICATION',
    'Control Flow Integrity',
    'VULNERABILITY SCAN',
    'PASSWORD STRENGTH METER',
    'PASSWORD MANAGEMENT',
    'PASSWORD POLICY',
    'GRAPHICAL AUTHENTICATION',
    'DATA LOSS PREVENTION',
    'DATA LEAKAGE DETECTION',
    'ACTIVITY MONITORING',
    'MOVING TARGET DEFENSE',
    'KEYSTROKE DYNAMICS',
    'ATTACK TREE',
    'Automatic Violation Prevention',
    'DISTRIBUTED LEDGER',
    'SOURCE IDENTIFICATION',
    'IMAGE RECOGNITION',
    'MAUVE',
    'HYPERGAME',
    'PREBUNKING',
    'NATURAL LANGUAGE PROCESSING',
    'LLM'
]

# Pre-compiled regex patterns for faster matching - VULNERABILITIES
VULNERABILITY_PATTERNS = {
    'DDoS': re.compile(r'ddos|denial[\s-]of[\s-]service|distributed[\s-]denial', re.I),
    'Phishing': re.compile(r'phishing|credential[\s-]harvesting|spear[\s-]phishing', re.I),
    'Ransomware': re.compile(r'ransomware|wannacry|lockbit|cryptolocker|wannacry|cryptolocker|colonial[\s-]pipeline', re.I),
    'Password_Attack': re.compile(r'password[\s-]attack|credential[\s-]stuffing|password[\s-]spraying|brute[\s-]force', re.I),
    'SQL_Injection': re.compile(r'sql[\s-]injection|sql[\s-]i|sqli', re.I),
    'Account_Hijacking': re.compile(r'account[\s-]hijack|account[\s-]takeover|ato', re.I),
    'Defacement': re.compile(r'defacement|website[\s-]hack|web[\s-]attack', re.I),
    'Trojan': re.compile(r'trojan|dropper[\s-]trojan', re.I),
    'Vulnerability': re.compile(r'vulnerability[\s-]exploit|cve[\s-]\d{4}[\s-]\d+|security[\s-]advisory', re.I),
    'Zero-day': re.compile(r'zero[\s-]day|0[\s-]day|\b0day\b', re.I),
    'APT': re.compile(r'\bapt\b|advanced[\s-]persistent[\s-]threat', re.I),
    'XSS': re.compile(r'cross[\s-]site[\s-]scripting|xss', re.I),
    'Malware': re.compile(r'malware|trojan|spyware|malvertis|ransomware|computer[\s-]virus|worm|keylog|keystroke[\s-]log|malicious[\s-]code|malicious[\s-]software|adware|rootkit|botnet|backdoor', re.I),
    'Data_Breach': re.compile(r'data[\s-]breach|data[\s-]leak|information[\s-]leak|data[\s-]spill|unintentional[\s-]information[\s-]disclos', re.I),
    'Disinformation': re.compile(r'disinformation|misinformation|false[\s-]information|misleading[\s-]information|deepfake', re.I),
    'Targeted_Attack': re.compile(r'targeted[\s-]attack', re.I),
    'Adware': re.compile(r'adware|180search[\s-]assistant', re.I),
    'Brute_Force': re.compile(r'brute[\s-]force', re.I),
    'Malvertising': re.compile(r'malvertis', re.I),
    'Backdoor': re.compile(r'backdoor', re.I),
    'Botnet': re.compile(r'botnet', re.I),
    'Cryptojacking': re.compile(r'cryptojack', re.I),
    'Worm': re.compile(r'worm[\s-]attack', re.I),
    'Spyware': re.compile(r'spyware|pegasus|gator[\s-]gain|coolwebsearch|transponder[\s-]vx2', re.I),
    'MITM': re.compile(r'man[\s-]in[\s-]the[\s-]middle|mitm', re.I),
    'DNS_Spoofing': re.compile(r'dns[\s-]spoof|dns[\s-]cache[\s-]poison', re.I),
    'Insider_Threat': re.compile(r'insider[\s-]threat', re.I),
    'Drive-by': re.compile(r'drive[\s-]by[\s-]download|drive[\s-]by[\s-]install', re.I),
    'Rootkit': re.compile(r'rootkit', re.I),
    'Adversarial_Attack': re.compile(r'adversarial[\s-]attack', re.I),
    'Data_Poisoning': re.compile(r'data[\s-]poisoning', re.I),
    'Deepfake': re.compile(r'deepfake', re.I),
    'Deeplocker': re.compile(r'deeplocker|deep[\s-]locker', re.I),
    'Supply_Chain': re.compile(r'supply[\s-]chain[\s-]attack', re.I),
    'IoT_Attack': re.compile(r'iot[\s-]device[\s-]attack|iot[\s-]attack|attack[\s-]on[\s-]iot|attack[\s-]the[\s-]iot', re.I),
    'DNS_Tunneling': re.compile(r'dns[\s-]tunnel', re.I),
    'Session_Hijacking': re.compile(r'session[\s-]hijack|hijack[\s-]session|cookie[\s-]hijack|cookie[\s-]poison', re.I),
    'URL_Attack': re.compile(r'url[\s-]attack|url[\s-]manipul|url[\s-]tamper|url[\s-]interpret', re.I),
    'Unknown_Attack': re.compile(r'unknown[\s-]attack', re.I)
}

# Pre-compiled regex patterns for SOLUTIONS
SOLUTION_PATTERNS = {
    'BLOCKCHAIN': re.compile(r'blockchain|distributed.ledger', re.I),
    'ACCESS CONTROL': re.compile(r'access.control', re.I),
    'ENCRYPTION': re.compile(r'encryption|encrypt', re.I),
    'SUPPLY CHAIN RISK MANAGEMENT': re.compile(r'supply.chain.risk.management', re.I),
    'IDENTITY MANAGEMENT': re.compile(r'identity.management', re.I),
    'Double Patterning Lithography': re.compile(r'double.patterning.lithography', re.I),
    'MACHINE LEARNING': re.compile(r'machine.learning|ml.algorithm', re.I),
    'ANOMALY DETECTION': re.compile(r'anomaly.detection', re.I),
    'CRYPTOGRAPHY': re.compile(r'cryptography|cryptographic', re.I),
    'PENETRATION TESTING': re.compile(r'penetration.testing|pen.test', re.I),
    'INTRUSION DETECTION PREVENTION': re.compile(r'intrusion.detection|intrusion.prevention|ids|ips', re.I),
    'STATIC ANALYSIS': re.compile(r'static.analysis', re.I),
    'DYNAMIC ANALYSIS': re.compile(r'dynamic.analysis', re.I),
    'MULTI FACTOR AUTHENTICATION': re.compile(r'multi.factor.authentication|mfa|two.factor|2fa', re.I),
    'LEAST PRIVILEGE': re.compile(r'least.privilege', re.I),
    'SESSION MANAGEMENT': re.compile(r'session.management', re.I),
    'CAPTCHA': re.compile(r'captcha', re.I),
    'BLACKLISTING': re.compile(r'blacklisting', re.I),
    'RATE LIMITING': re.compile(r'rate.limiting', re.I),
    'GRAPHICAL MODEL': re.compile(r'graphical.model', re.I),
    'HONEYPOT': re.compile(r'honeypot', re.I),
    'SOFTWARE DEFINED NETWORK': re.compile(r'software.defined.network|sdn', re.I),
    'GAME THEORY': re.compile(r'game.theory', re.I),
    'GRAPH MACHINE LEARNING': re.compile(r'graph.machine.learning|graph.based.machine.learning', re.I),
    'IP WHITELIST': re.compile(r'ip.whitelist', re.I),
    'TRAFFIC SHAPING': re.compile(r'traffic.shaping', re.I),
    'PACKET FILTERING': re.compile(r'packet.filtering', re.I),
    'BLACKHOLING': re.compile(r'blackholing|black.holing', re.I),
    'RANK CORRELATION': re.compile(r'rank.correlation', re.I),
    'D3NS DNS': re.compile(r'd3ns|dns.security', re.I),
    'SESSION ID RANDOMIZATION': re.compile(r'session.id.randomization|session.id.randomisation', re.I),
    'STRONG AUTHENTICATION': re.compile(r'strong.authentication|kerberos', re.I),
    'SECURE SOCKETS LAYER': re.compile(r'secure.sockets.layer|transport.layer.security|tls|ssl', re.I),
    'HTTPS': re.compile(r'https', re.I),
    'CONTINUOUS AUTHENTICATION': re.compile(r'continuous.authentication', re.I),
    'Identity-Based Encryption': re.compile(r'identity.based.encryption|identity.based.encryption', re.I),
    'DATA SANITIZATION': re.compile(r'data.sanitization|data.sanitisation', re.I),
    'OUTLIER DETECTION': re.compile(r'outlier.detection', re.I),
    'DATA PROVENANCE': re.compile(r'data.provenance', re.I),
    'ADVERSARIAL TRAINING': re.compile(r'adversarial.training|adversarial.learning', re.I),
    'TRUSTWORTHY AI': re.compile(r'trustworthy.ai', re.I),
    'DEEP PROBABILISTIC MODEL': re.compile(r'deep.probabilistic.model', re.I),
    'Bayesian Network': re.compile(r'bayesian.network', re.I),
    'TROJAN ISOLATION': re.compile(r'trojan.isolation', re.I),
    'HARDWARE SANDBOXING': re.compile(r'hardware.sandboxing', re.I),
    'BEHAVIOR BASED DETECTION': re.compile(r'behavior.based.detection|behavior.based.detection|behaviour.based.detection|behaviour.based.detection', re.I),
    'FORMAL VERIFICATION': re.compile(r'formal.verification', re.I),
    'SPLIT MANUFACTURING': re.compile(r'split.manufacturing', re.I),
    'VULNERABILITY MANAGEMENT': re.compile(r'vulnerability.management', re.I),
    'FILE INTEGRITY MONITORING': re.compile(r'file.integrity.monitoring', re.I),
    'VPN': re.compile(r'vpn|virtual.private.network', re.I),
    'CSS MATCHING': re.compile(r'css.matching', re.I),
    'URI MATCHING': re.compile(r'uri.matching', re.I),
    'PRIVACY PRESERVING': re.compile(r'privacy.preserving', re.I),
    'SECURE BOOT': re.compile(r'secure.boot', re.I),
    'MERKLE SIGNATURE': re.compile(r'merkle.signature', re.I),
    'LIVENESS DETECTION': re.compile(r'liveness.detection', re.I),
    'AUDIO DEEPFAKE DETECTION': re.compile(r'audio.deepfake.detection', re.I),
    '3D FACE RECONSTRUCTION': re.compile(r'3d.face.reconstruction', re.I),
    'BIOMETRICS': re.compile(r'biometrics', re.I),
    'DIGITAL WATERMARK': re.compile(r'digital.watermark', re.I),
    'APPLICATION WHITELISTING': re.compile(r'application.whitelisting', re.I),
    'DATA BACKUP': re.compile(r'data.backup', re.I),
    'HIDDEN MARKOV MODEL': re.compile(r'hidden.markov.model', re.I),
    'PATCH MANAGEMENT': re.compile(r'patch.management', re.I),
    'DATA AUGMENTATION': re.compile(r'data.augmentation', re.I),
    'DIMENSIONALITY REDUCTION': re.compile(r'dimensionality.reduction', re.I),
    'DEFENSIVE DISTILLATION': re.compile(r'defensive.distillation', re.I),
    'GRADIENT MASKING': re.compile(r'gradient.masking', re.I),
    'RRAM': re.compile(r'rram', re.I),
    'SPATIAL SMOOTHING': re.compile(r'spatial.smoothing', re.I),
    'NOISE INJECTION': re.compile(r'noise.injection', re.I),
    'TAINT ANALYSIS': re.compile(r'taint.analysis', re.I),
    'NETWORK SEGMENTATION': re.compile(r'network.segmentation', re.I),
    'USER BEHAVIOR ANALYTICS': re.compile(r'user.behavior.analytics|user.behaviour.analytics', re.I),
    'DECEPTION TECHNOLOGY': re.compile(r'deception.technology', re.I),
    'RISK ASSESSMENT': re.compile(r'risk.assessment', re.I),
    'LOG CORRELATION': re.compile(r'log.correlation', re.I),
    'DYNAMIC RESOURCE MANAGEMENT': re.compile(r'dynamic.resource.management', re.I),
    'SANDBOXING': re.compile(r'sandboxing', re.I),
    'DARKNET MONITORING': re.compile(r'darknet.monitoring|monitoring.darknet', re.I),
    'VIRTUAL KEYBOARD': re.compile(r'virtual.keyboard', re.I),
    'CODE SIGN': re.compile(r'code.sign', re.I),
    'FILE SIGNATURE': re.compile(r'file.signature', re.I),
    'PUBLIC KEY INFRASTRUCTURE': re.compile(r'public.key.infrastructure|pki', re.I),
    'VOICEPRINT AUTHENTICATION': re.compile(r'voiceprint.authentication', re.I),
    'MUTUAL AUTHENTICATION': re.compile(r'mutual.authentication', re.I),
    'PASSWORD SALT': re.compile(r'password.salt', re.I),
    'ONE TIME PASSWORD': re.compile(r'one.time.password|otp', re.I),
    'DYNAMIC BINARY INSTRUMENTATION': re.compile(r'dynamic.binary.instrumentation', re.I),
    'ORTHOGONAL OBFUSCATION': re.compile(r'orthogonal.obfuscation', re.I),
    'PASSWORD HASH': re.compile(r'password.hash', re.I),
    'DNSSEC': re.compile(r'dnssec', re.I),
    'CERTIFICATE PINNING': re.compile(r'certificate.pinning', re.I),
    'SECURE SIMPLE PAIRING': re.compile(r'secure.simple.pairing', re.I),
    'VULNERABILITY ASSESSMENT': re.compile(r'vulnerability.assessment', re.I),
    'SIEM': re.compile(r'siem', re.I),
    'STANDARDIZED COMMUNICATION': re.compile(r'standardized.communication|standardised.communication', re.I),
    'Control Flow Integrity': re.compile(r'control.flow.integrity', re.I),
    'VULNERABILITY SCAN': re.compile(r'vulnerability.scan', re.I),
    'PASSWORD STRENGTH METER': re.compile(r'password.strength.meter', re.I),
    'PASSWORD MANAGEMENT': re.compile(r'password.management', re.I),
    'PASSWORD POLICY': re.compile(r'password.policy', re.I),
    'GRAPHICAL AUTHENTICATION': re.compile(r'graphical.authentication', re.I),
    'DATA LOSS PREVENTION': re.compile(r'data.loss.prevention|dlp', re.I),
    'DATA LEAKAGE DETECTION': re.compile(r'data.leakage.detection|data.leakage.prevention', re.I),
    'ACTIVITY MONITORING': re.compile(r'activity.monitoring', re.I),
    'MOVING TARGET DEFENSE': re.compile(r'moving.target.defense|moving.target.defence', re.I),
    'KEYSTROKE DYNAMICS': re.compile(r'keystroke.dynamics', re.I),
    'ATTACK TREE': re.compile(r'attack.tree', re.I),
    'Automatic Violation Prevention': re.compile(r'automatic.violation.prevention', re.I),
    'DISTRIBUTED LEDGER': re.compile(r'distributed.ledger', re.I),
    'SOURCE IDENTIFICATION': re.compile(r'source.identification', re.I),
    'IMAGE RECOGNITION': re.compile(r'image.recognition', re.I),
    'MAUVE': re.compile(r'mauve', re.I),
    'HYPERGAME': re.compile(r'hypergame', re.I),
    'PREBUNKING': re.compile(r'prebunking', re.I),
    'NATURAL LANGUAGE PROCESSING': re.compile(r'natural.language.processing|language.model|llm', re.I),
    'LLM': re.compile(r'llm|large.language.model', re.I)
}

# Date configuration
START_DATE = datetime(2011, 7, 1)
END_DATE = datetime(2024, 12, 31)

class OptimizedRSSProcessor:
    def __init__(self):
        # Pre-calculate date mappings for O(1) lookups
        self.date_range = pd.date_range(START_DATE, END_DATE, freq='D')
        self.date_to_index = {date.date(): idx for idx, date in enumerate(self.date_range)}
        self.date_strings = [d.strftime('%d/%m/%Y') for d in self.date_range]
        
        # Initialize count arrays for both vulnerabilities and solutions
        self.vulnerability_counts = {cat: np.zeros(len(self.date_range), dtype=np.int32) 
                                   for cat in SECURITY_CATEGORIES}
        self.solution_counts = {sol: np.zeros(len(self.date_range), dtype=np.int32) 
                              for sol in SOLUTIONS_LIST}
        self.all_vulnerability_counts = np.zeros(len(self.date_range), dtype=np.int32)
        self.all_solution_counts = np.zeros(len(self.date_range), dtype=np.int32)
        
    async def fetch_feed_async(self, session: aiohttp.ClientSession, url: str) -> List:
        """Async feed fetching with timeout and error handling."""
        try:
            timeout = aiohttp.ClientTimeout(total=10)
            async with session.get(url, timeout=timeout) as response:
                if response.status == 200:
                    content = await response.text()
                    feed = feedparser.parse(content)
                    return feed.entries if hasattr(feed, 'entries') else []
                else:
                    print(f"HTTP {response.status} for {url}")
                    return []
        except Exception as e:
            print(f"Error fetching {url}: {str(e)}")
            return []
    
    async def fetch_all_feeds_async(self) -> List:
        """Fetch all feeds concurrently using async I/O."""
        connector = aiohttp.TCPConnector(limit=20, limit_per_host=5)
        timeout = aiohttp.ClientTimeout(total=30)
        
        async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
            tasks = [self.fetch_feed_async(session, url) for url in FEEDS]
            results = await asyncio.gather(*tasks, return_exceptions=True)
            
            # Flatten results and filter out exceptions
            all_entries = []
            for result in results:
                if isinstance(result, list):
                    all_entries.extend(result)
                elif isinstance(result, Exception):
                    print(f"Task failed: {result}")
            
            return all_entries
    
    @lru_cache(maxsize=2000)
    def categorize_vulnerability_cached(self, title: str, summary: str) -> str:
        """Cached vulnerability categorization with optimized text processing."""
        if not title and not summary:
            return 'Unknown'
        
        # Combine and normalize text once
        text = f"{title or ''} {summary or ''}".lower()
        
        # Early exit for common patterns
        if 'vulnerability' in text or 'cve-' in text:
            return 'Vulnerability'
        if 'ransomware' in text:
            return 'Ransomware'
        if 'phishing' in text:
            return 'Phishing'
        
        # Check remaining patterns
        for category, pattern in VULNERABILITY_PATTERNS.items():
            if pattern.search(text):
                return category
        
        return 'Unknown'
    
    @lru_cache(maxsize=2000)
    def categorize_solution_cached(self, title: str, summary: str) -> List[str]:
        """Cached solution categorization - returns list of matching solutions."""
        if not title and not summary:
            return []
        
        # Combine and normalize text once
        text = f"{title or ''} {summary or ''}".lower()
        
        matching_solutions = []
        for solution, pattern in SOLUTION_PATTERNS.items():
            if pattern.search(text):
                matching_solutions.append(solution)
        
        return matching_solutions
    
    def parse_date_fast(self, published_parsed) -> datetime:
        """Fast date parsing with error handling."""
        try:
            if published_parsed and len(published_parsed) >= 6:
                return datetime(*published_parsed[:6])
        except (ValueError, TypeError, OverflowError):
            pass
        return None
    
    def process_entries_batch(self, entries: List) -> None:
        """Process entries in batches for better memory efficiency."""
        print(f"Processing {len(entries)} entries...")
        
        # Filter valid entries first
        valid_entries = []
        for entry in entries:
            if hasattr(entry, 'published_parsed'):
                pub_date = self.parse_date_fast(entry.published_parsed)
                if pub_date and pub_date.date() in self.date_to_index:
                    valid_entries.append((entry, pub_date))
        
        print(f"Found {len(valid_entries)} valid entries in date range")
        
        # Process in batches to reduce memory pressure
        batch_size = 1000
        for i in range(0, len(valid_entries), batch_size):
            batch = valid_entries[i:i + batch_size]
            self.process_batch(batch)
            
            if i % 5000 == 0 and i > 0:
                print(f"Processed {i}/{len(valid_entries)} entries...")
    
    def process_batch(self, batch: List[Tuple]) -> None:
        """Process a batch of entries efficiently."""
        for entry, pub_date in batch:
            try:
                idx = self.date_to_index[pub_date.date()]
                
                # Get text content
                title = getattr(entry, 'title', '') or ''
                summary = getattr(entry, 'summary', '') or ''
                
                # Categorize vulnerabilities
                vulnerability_category = self.categorize_vulnerability_cached(title, summary)
                
                # Update vulnerability counts
                if vulnerability_category in self.vulnerability_counts:
                    self.vulnerability_counts[vulnerability_category][idx] += 1
                    self.all_vulnerability_counts[idx] += 1
                
                # Categorize solutions
                solution_categories = self.categorize_solution_cached(title, summary)
                
                # Update solution counts
                for solution_category in solution_categories:
                    if solution_category in self.solution_counts:
                        self.solution_counts[solution_category][idx] += 1
                        self.all_solution_counts[idx] += 1
                    
            except Exception as e:
                continue  # Skip problematic entries
    
    def save_counts_fast(self, 
                        vulnerability_file: str = 'security_RSS_vulnerabilities_daily.csv',
                        solution_file: str = 'security_RSS_solutions_daily.csv') -> Tuple[str, str]:
        """Fast CSV generation using pandas with dates as row headers."""
        print("Saving daily counts...")
        
        # Save vulnerabilities - dates as rows, categories as columns
        vuln_data = {'Date': self.date_strings}
        for category in SECURITY_CATEGORIES:
            vuln_data[category] = self.vulnerability_counts[category]
        vuln_data['ALL_VULNERABILITIES'] = self.all_vulnerability_counts
        
        vuln_df = pd.DataFrame(vuln_data)
        vuln_df.to_csv(vulnerability_file, index=False)
        print(f"Vulnerability counts saved to {vulnerability_file}")
        
        # Save solutions - dates as rows, solutions as columns
        sol_data = {'Date': self.date_strings}
        for solution in SOLUTIONS_LIST:
            sol_data[solution] = self.solution_counts[solution]
        sol_data['ALL_SOLUTIONS'] = self.all_solution_counts
        
        sol_df = pd.DataFrame(sol_data)
        sol_df.to_csv(solution_file, index=False)
        print(f"Solution counts saved to {solution_file}")
        
        return vulnerability_file, solution_file
    
    def aggregate_to_monthly_fast(self, 
                                 input_file: str, 
                                 output_file: str,
                                 file_type: str = 'vulnerabilities') -> None:
        """Fast monthly aggregation with dates as row headers."""
        print(f"Aggregating {file_type} to monthly counts...")
        
        # Read and process efficiently
        df = pd.read_csv(input_file)
        df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
        
        # Set index and resample in one operation
        monthly = df.set_index('Date').resample('M').sum()
        
        # Format dates as MM/YYYY for row headers
        monthly.index = monthly.index.strftime('%m/%Y')
        
        # Reset index to make dates a column (row headers in CSV)
        monthly.reset_index(inplace=True)
        monthly.rename(columns={'index': 'Date'}, inplace=True)
        
        monthly.to_csv(output_file, index=False)
        print(f"Monthly {file_type} aggregation saved to {output_file}")

async def main():
    """Main execution function with performance monitoring."""
    print("Starting optimized RSS security feeds processing...")
    start_time = time.time()
    
    processor = OptimizedRSSProcessor()
    
    # Fetch feeds asynchronously
    print("Fetching RSS feeds asynchronously...")
    fetch_start = time.time()
    entries = await processor.fetch_all_feeds_async()
    fetch_time = time.time() - fetch_start
    print(f"Fetched {len(entries)} entries in {fetch_time:.2f} seconds")
    
    # Process entries
    process_start = time.time()
    processor.process_entries_batch(entries)
    process_time = time.time() - process_start
    print(f"Processed entries in {process_time:.2f} seconds")
    
    # Save results
    save_start = time.time()
    vuln_file, sol_file = processor.save_counts_fast()
    
    # Create monthly aggregations
    processor.aggregate_to_monthly_fast(vuln_file, 'security_RSS_vulnerabilities_monthly.csv', 'vulnerabilities')
    processor.aggregate_to_monthly_fast(sol_file, 'security_RSS_solutions_monthly.csv', 'solutions')
    
    save_time = time.time() - save_start
    print(f"Saved results in {save_time:.2f} seconds")
    
    total_time = time.time() - start_time
    print(f"\nTotal processing time: {total_time:.2f} seconds")
    print(f"Performance breakdown:")
    print(f"  - Fetching: {fetch_time:.2f}s ({fetch_time/total_time*100:.1f}%)")
    print(f"  - Processing: {process_time:.2f}s ({process_time/total_time*100:.1f}%)")
    print(f"  - Saving: {save_time:.2f}s ({save_time/total_time*100:.1f}%)")

def run_optimized():
    """Entry point for the optimized processor."""
    # Check if required packages are available
    try:
        import aiohttp
    except ImportError:
        print("Installing required packages...")
        import subprocess
        subprocess.check_call(["pip", "install", "aiohttp"])
        import aiohttp
    
    # Run the async main function
    asyncio.run(main())

if __name__ == '__main__':
    run_optimized()

In [ ]:
import pandas as pd
from datetime import datetime
import io

input_data = "Cyber Events Database - 2014-2024 + Jan & Aug 2025.csv"


# Country code mapping
COUNTRY_CODES = {
    'United States of America': 'US',
    'United States': 'US',
    'United Kingdom': 'GB',
    'Great Britain': 'GB',
    'UK': 'GB',
    'Canada': 'CA',
    'Australia': 'AU',
    'Ukraine': 'UA',
    'Russia': 'RU',
    'Russian Federation': 'RU',
    'France': 'FR',
    'Germany': 'DE',
    'Brazil': 'BR',
    'China': 'CN',
    'Japan': 'JP',
    'Pakistan': 'PK',
    'North Korea': 'KP',
    'South Korea': 'KR',
    'Korea, Republic of': 'KR',
    'India': 'IN',
    'Taiwan': 'TW',
    'Netherlands': 'NL',
    'Spain': 'ES',
    'Sweden': 'SE',
    'Mexico': 'MX',
    'Iran': 'IR',
    'Israel': 'IL',
    'Saudi Arabia': 'SA',
    'Syria': 'SY',
    'Syrian Arab Republic': 'SY',
    'Finland': 'FI',
    'Ireland': 'IE',
    'Austria': 'AT',
    'Norway': 'NO',
    'Switzerland': 'CH',
    'Italy': 'IT',
    'Malaysia': 'MY',
    'Egypt': 'EG',
    'Turkey': 'TR',
    'Portugal': 'PT',
    'Palestine': 'PS',
    'United Arab Emirates': 'AE'
}

# Target country codes (including '?' for undetermined and 'ALL' for total)
TARGET_CODES = ['US','GB','CA','AU','UA','RU','FR','DE','BR','CN','JP','PK',
               'KP','KR','IN','TW','NL','ES','SE','MX','IR','IL','SA','SY',
               'FI','IE','AT','NO','CH','IT','MY','EG','TR','PT','PS','AE','?','ALL']

# Define all unique event subtypes from your data
EVENT_SUBTYPES = [
    'Exploitation of Application Server',
    'Data Attack',
    'External Denial of Service',
    'Exploitation of End Hosts',
    'Message Manipulation',
    'Undetermined',
    'Exploitation of Sensors',
    'Exploitation of Network Infrastructure',
    'Internal Denial of Service',
    'Physical Attack',
    'Exploitation of Data in Transit',
    'Exploitation of Network Server'
]

def map_country_code(country_name):
    """Map country name to country code, return '?' for undetermined"""
    if pd.isna(country_name) or country_name == 'Undetermined' or country_name == '':
        return '?'
    
    country_name = str(country_name).strip()
    return COUNTRY_CODES.get(country_name, '?')

def parse_event_subtypes(event_subtype):
    """Parse event subtype string and return list of individual subtypes"""
    if pd.isna(event_subtype) or event_subtype == '':
        return ['Undetermined']
    
    # Split by comma and strip whitespace
    subtypes = [subtype.strip() for subtype in str(event_subtype).split(',')]
    
    # Handle special cases and normalize
    normalized_subtypes = []
    for subtype in subtypes:
        # Fix common typos/variations
        if subtype == 'Exploitation of Application Servers':
            subtype = 'Exploitation of Application Server'
        elif subtype == 'Exploitation of  Network Infrastructure':
            subtype = 'Exploitation of Network Infrastructure'
        normalized_subtypes.append(subtype)
    
    return normalized_subtypes if normalized_subtypes else ['Undetermined']

def read_csv_with_encoding(file_path):
    """Try different encodings to read the CSV file"""
    encodings = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252', 'windows-1252']
    
    for encoding in encodings:
        try:
            print(f"Trying encoding: {encoding}")
            df = pd.read_csv(file_path, encoding=encoding, sep=',')
            print(f"Successfully read file with {encoding} encoding")
            return df
        except UnicodeDecodeError as e:
            print(f"Failed with {encoding}: {e}")
            continue
        except Exception as e:
            print(f"Error with {encoding}: {e}")
            continue
    
    # If all encodings fail, try with error handling
    try:
        print("Trying with error handling...")
        df = pd.read_csv(file_path, encoding='utf-8', sep=',', errors='ignore')
        print("Successfully read file with error handling")
        return df
    except Exception as e:
        print(f"Final attempt failed: {e}")
        raise

def process_cyber_attacks_data(df):
    """Process the cyber attacks data and generate the output format"""
    
    # Map country codes
    df['country_code'] = df['country'].apply(map_country_code)
    
    # Parse event subtypes
    df['event_subtypes_list'] = df['event_subtype'].apply(parse_event_subtypes)
    
    # Create date string for grouping (Month-Year)
    df['date_str'] = df['month'].astype(str) + '-' + df['year'].astype(str)
    
    # Convert to proper date format for sorting
    df['date'] = pd.to_datetime(df['year'].astype(str) + '-' + df['month'].astype(str) + '-01')
    
    # Get unique dates sorted chronologically
    unique_dates = sorted(df['date'].unique())
    
    # Create result dataframe
    result_rows = []
    
    for date in unique_dates:
        month_year = date.strftime('%B-%y')
        year_month = date.strftime('%Y-%m')
        
        # Filter data for this month
        month_data = df[df['date'] == date]
        
        row_data = {'Date': month_year}
        
        # Count events for each country code and event subtype combination
        for country_code in TARGET_CODES:
            if country_code == 'ALL':
                # Count all subtypes for all countries
                all_subtypes_count = {}
                for subtypes_list in month_data['event_subtypes_list']:
                    for subtype in subtypes_list:
                        all_subtypes_count[subtype] = all_subtypes_count.get(subtype, 0) + 1
                
                # Add counts for each subtype
                for subtype in EVENT_SUBTYPES:
                    column_name = f'{subtype}-{country_code}'
                    row_data[column_name] = all_subtypes_count.get(subtype, 0)
            else:
                # Count for specific country code
                country_data = month_data[month_data['country_code'] == country_code]
                
                # Count each subtype for this country
                subtype_counts = {}
                for subtypes_list in country_data['event_subtypes_list']:
                    for subtype in subtypes_list:
                        subtype_counts[subtype] = subtype_counts.get(subtype, 0) + 1
                
                # Add counts for each subtype
                for subtype in EVENT_SUBTYPES:
                    column_name = f'{subtype}-{country_code}'
                    row_data[column_name] = subtype_counts.get(subtype, 0)
        
        result_rows.append(row_data)
    
    # Create final dataframe
    result_df = pd.DataFrame(result_rows)
    
    # Create column order: Date first, then all subtype-country combinations
    columns = ['Date']
    for subtype in EVENT_SUBTYPES:
        for country_code in TARGET_CODES:
            columns.append(f'{subtype}-{country_code}')
    
    # Only include columns that exist in the result dataframe
    available_columns = [col for col in columns if col in result_df.columns]
    result_df = result_df[available_columns]
    
    return result_df

def create_summary_by_attack_type(df):
    """Create a simplified summary by attack type only (without country breakdown)"""
    
    # Parse event subtypes
    df['event_subtypes_list'] = df['event_subtype'].apply(parse_event_subtypes)
    
    # Convert to proper date format for sorting
    df['date'] = pd.to_datetime(df['year'].astype(str) + '-' + df['month'].astype(str) + '-01')
    
    # Get unique dates sorted chronologically
    unique_dates = sorted(df['date'].unique())
    
    # Create result dataframe
    result_rows = []
    
    for date in unique_dates:
        month_year = date.strftime('%B-%y')
        
        # Filter data for this month
        month_data = df[df['date'] == date]
        
        row_data = {'Date': month_year}
        
        # Count each subtype
        subtype_counts = {}
        for subtypes_list in month_data['event_subtypes_list']:
            for subtype in subtypes_list:
                subtype_counts[subtype] = subtype_counts.get(subtype, 0) + 1
        
        # Add counts for each subtype
        for subtype in EVENT_SUBTYPES:
            row_data[subtype] = subtype_counts.get(subtype, 0)
        
        # Add total count
        row_data['Total'] = len(month_data)
        
        result_rows.append(row_data)
    
    return pd.DataFrame(result_rows)

# Main execution
if __name__ == "__main__":
    try:
        # Read the CSV file with encoding handling
        file_path = "Cyber Events Database - 2014-2024 + Jan & Aug 2025.csv"
        df = read_csv_with_encoding(file_path)
        
        # Display basic info about the data
        print(f"Data shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
        print(f"Date range: {df['year'].min()} - {df['year'].max()}")
        
        # Process the data with detailed breakdown
        print("\nProcessing detailed cyber attacks data...")
        detailed_result = process_cyber_attacks_data(df)
        
        # Display first few rows of detailed results
        print("\nDetailed Cyber Attacks Data (first few columns):")
        print(detailed_result.iloc[:, :10].to_string(index=False))  # Show first 10 columns
        
        # Create summary by attack type only
        print("\nCreating summary by attack type...")
        summary_result = create_summary_by_attack_type(df)
        print("\nSummary by Attack Type:")
        print(summary_result.to_string(index=False))
        
        # Save to CSV files
        detailed_result.to_csv('cyber_attacks_detailed_breakdown.csv', index=False)
        summary_result.to_csv('cyber_attacks_summary_by_type.csv', index=False)
        
        print(f"\nDetailed data saved to 'cyber_attacks_detailed_breakdown.csv'")
        print(f"Summary data saved to 'cyber_attacks_summary_by_type.csv'")
        print(f"Detailed data shape: {detailed_result.shape}")
        print(f"Summary data shape: {summary_result.shape}")
        
    except Exception as e:
        print(f"Error processing file: {e}")
        print("\nTroubleshooting steps:")
        print("1. Check if the file exists at the specified path")
        print("2. Try specifying the full path to the file")
        print("3. Check if the file is open in another program")
        print("4. Try opening the file in Excel and saving as UTF-8 CSV")

# Alternative function for manual encoding specification
def process_with_specific_encoding(file_path, encoding='latin-1'):
    """Process file with specific encoding"""
    try:
        df = pd.read_csv(file_path, encoding=encoding, sep=',')
        detailed_result = process_cyber_attacks_data(df)
        summary_result = create_summary_by_attack_type(df)
        
        detailed_result.to_csv('cyber_attacks_detailed_breakdown.csv', index=False)
        summary_result.to_csv('cyber_attacks_summary_by_type.csv', index=False)
        
        return detailed_result, summary_result
    except Exception as e:
        print(f"Error with {encoding} encoding: {e}")
        return None, None



1. Download the Cyber Events Database from https://cissm.umd.edu/research-impact/publications/cyber-events-database-home
2. Place the file at the top of the directory

In [ ]:
import pandas as pd
import calendar
from datetime import datetime

# Read the CSV file
df = pd.read_csv('number_of_political_violence_events_by_country-month-year_as-of-26Sep2025.csv')

# Filter data for January 2017 to December 2024
df = df[(df['YEAR'] >= 2017) & (df['YEAR'] <= 2024)]

# Convert month names to numbers for proper sorting
month_map = {month: index for index, month in enumerate(calendar.month_name) if month}
df['MONTH_NUM'] = df['MONTH'].map(month_map)

# Create a proper date column for sorting
df['DATE'] = pd.to_datetime(df['YEAR'].astype(str) + '-' + df['MONTH_NUM'].astype(str) + '-01')

# Sort by date
df = df.sort_values('DATE')

# Define the specific countries in the exact order requested
countries_order = [
    'US', 'GB', 'CA', 'AU', 'UA', 'RU', 'FR', 'DE', 'BR', 'CN', 
    'JP', 'PK', 'KP', 'KR', 'IN', 'TW', 'NL', 'ES', 'SE', 'MX', 
    'IR', 'IL', 'SA', 'SY', 'FI', 'IE', 'AT', 'NO', 'CH', 'IT', 
    'MY', 'EG', 'TR', 'PT', 'PS', 'AE'
]

# Map of country codes to full country names (you may need to adjust based on your data)
country_name_map = {
    'US': 'United States',
    'GB': 'United Kingdom',
    'CA': 'Canada',
    'AU': 'Australia',
    'UA': 'Ukraine',
    'RU': 'Russia',
    'FR': 'France',
    'DE': 'Germany',
    'BR': 'Brazil',
    'CN': 'China',
    'JP': 'Japan',
    'PK': 'Pakistan',
    'KP': 'North Korea',
    'KR': 'South Korea',
    'IN': 'India',
    'TW': 'Taiwan',
    'NL': 'Netherlands',
    'ES': 'Spain',
    'SE': 'Sweden',
    'MX': 'Mexico',
    'IR': 'Iran',
    'IL': 'Israel',
    'SA': 'Saudi Arabia',
    'SY': 'Syria',
    'FI': 'Finland',
    'IE': 'Ireland',
    'AT': 'Austria',
    'NO': 'Norway',
    'CH': 'Switzerland',
    'IT': 'Italy',
    'MY': 'Malaysia',
    'EG': 'Egypt',
    'TR': 'Turkey',
    'PT': 'Portugal',
    'PS': 'Palestine',
    'AE': 'United Arab Emirates'
}

# Pivot the data to get countries as columns
pivot_df = df.pivot_table(
    index=['YEAR', 'MONTH_NUM', 'MONTH', 'DATE'], 
    columns='COUNTRY', 
    values='EVENTS', 
    aggfunc='sum',
    fill_value=0
).reset_index()

# Create the Date column in "MMM-YY" format
pivot_df['Date'] = pivot_df['MONTH'].str[:3] + '-' + pivot_df['YEAR'].astype(str).str[2:]

# Calculate worldwide total per month
all_countries = [col for col in pivot_df.columns if col not in ['YEAR', 'MONTH_NUM', 'MONTH', 'DATE', 'Date']]
pivot_df['Holidays (ALL)'] = pivot_df[all_countries].sum(axis=1)

# Create columns for each requested country (fill with 0 if country not in data)
for country_code in countries_order:
    country_full_name = country_name_map.get(country_code, country_code)
    if country_full_name in pivot_df.columns:
        pivot_df[f'{country_code}_holiday'] = pivot_df[country_full_name]
    else:
        pivot_df[f'{country_code}_holiday'] = 0

# Select and reorder columns in the exact order requested
final_columns = ['Date', 'Holidays (ALL)'] + [f'{country}_holiday' for country in countries_order]

# Create final dataframe with the exact column order
final_df = pivot_df[final_columns].copy()

# Sort by year and month
final_df = final_df.sort_values(['Date'])

# Reset index
final_df = final_df.reset_index(drop=True)

# Display the first few rows to verify
print("First 12 months of data:")
print(final_df.head(12))

print(f"\nData shape: {final_df.shape}")
print(f"Date range: {final_df['Date'].iloc[0]} to {final_df['Date'].iloc[-1]}")
print(f"Total months: {len(final_df)}")

# Save to CSV
output_filename = 'political_violence_by_country_2017_2024.csv'
final_df.to_csv(output_filename, index=False)
print(f"\nData saved to: {output_filename}")

# Display summary statistics
print(f"\nSummary Statistics:")
print(f"Worldwide incidents - Min: {final_df['Holidays (ALL)'].min()}, Max: {final_df['Holidays (ALL)'].max()}, Avg: {final_df['Holidays (ALL)'].mean():.1f}")

# Check which countries have non-zero data
non_zero_countries = []
for country in countries_order:
    col_name = f'{country}_holiday'
    if final_df[col_name].sum() > 0:
        non_zero_countries.append(country)

print(f"Countries with non-zero data: {non_zero_countries}")

1. Download the number of political violence file from the ACLED
2. Place the file at the top of the directory

In [ ]:
"""

This script counts for each country the number of posts about wars and conflicts related to that country.
The output is the monthly count of these posts for each country in the period between January 2023 and December 2024.

Output file: ACA.csv
"""

import requests
import os
import json
import pandas as pd
import csv
import datetime
import dateutil.parser
import time
from csv import writer
from datetime import datetime, timedelta
from atproto import Client, models

# Bluesky API configuration - YOU NEED TO SET THESE CREDENTIALS
BLUESKY_HANDLE = " "  # Replace with your Bluesky handle
BLUESKY_APP_PASSWORD = " "     # Replace with your app password     

def setup_bluesky_client():
    """Set up and authenticate with Bluesky"""
    try:
        client = Client()
        client.login(BLUESKY_HANDLE, BLUESKY_APP_PASSWORD)
        print("Successfully authenticated with Bluesky")
        return client
    except Exception as e:
        print(f"Authentication failed: {e}")
        return None

def search_bluesky_posts(client, keyword, limit=100):
    """
    Search Bluesky posts using the correct API method
    """
    try:
        # Use the app.bsky.feed.searchPosts method correctly
        response = client.app.bsky.feed.search_posts({'q': keyword, 'limit': limit})
        return len(response.posts), response.posts
    except Exception as e:
        print(f"Search error for '{keyword}': {e}")
        return 0, []

def get_bluesky_timeline_posts(client, limit=100):
    """
    Get posts from the user's timeline as an alternative approach
    """
    try:
        response = client.get_timeline(limit=limit)
        return len(response.feed), response.feed
    except Exception as e:
        print(f"Timeline error: {e}")
        return 0, []

def count_keywords_in_posts(posts, keywords):
    """
    Count how many posts contain the target keywords
    """
    count = 0
    for post in posts:
        if hasattr(post, 'post') and hasattr(post.post, 'record'):
            text = post.post.record.text.lower()
        elif hasattr(post, 'record'):
            text = post.record.text.lower()
        else:
            continue
            
        for keyword in keywords:
            if keyword in text:
                count += 1
                break  # Count post only once even if multiple keywords match
    return count

def is_leap_year(year):
    """Determine whether a year is a leap year."""
    return year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)

def alternative_count_approach(client, country_terms, year, month):
    """
    Alternative approach to count posts using multiple search strategies
    """
    # Clean up country terms
    clean_terms = country_terms.replace('(', '').replace(')', '').replace(' OR ', ' ').split()
    
    # Create search combinations
    search_combinations = []
    
    # Single word searches for the country
    for term in clean_terms:
        if len(term) > 2:  # Avoid very short terms
            search_combinations.extend([
                f"{term} war",
                f"{term} conflict", 
                f"{term} military",
                f"{term} army",
                f"{term} attack",
                f"{term} defense"
            ])
    
    # Multi-word searches (take first 2 terms to avoid too long queries)
    if len(clean_terms) >= 2:
        main_terms = ' '.join(clean_terms[:2])
        search_combinations.extend([
            f"{main_terms} war",
            f"{main_terms} conflict",
            f'"{main_terms}" war',
            f'"{main_terms}" conflict'
        ])
    
    total_count = 0
    unique_posts = set()  # Track unique post IDs to avoid duplicates
    
    for search_query in search_combinations[:10]:  # Limit to 10 searches per country
        try:
            count, posts = search_bluesky_posts(client, search_query, limit=50)
            
            # Count unique posts
            for post in posts:
                if hasattr(post, 'uri'):
                    post_id = post.uri
                elif hasattr(post, 'post') and hasattr(post.post, 'uri'):
                    post_id = post.post.uri
                else:
                    continue
                    
                if post_id not in unique_posts:
                    unique_posts.add(post_id)
                    total_count += 1
            
            print(f"  Search '{search_query}': found {count} posts, unique: {len(unique_posts)}")
            time.sleep(1)  # Rate limiting
            
        except Exception as e:
            print(f"  Error searching '{search_query}': {e}")
            continue
    
    return total_count

# Country configurations
months = ['01','02','03','04','05','06','07','08','09','10','11','12']

countries_s = [
    'USA America', 'UK British United Kingdom Britain',
    'CANADA CANADIAN', 'AUSTRALIA', 'Ukraine', 'RUSSIA',
    'FRANCE FRENCH', 'GERMAN', 'Brazil', 'China chinese',
    'Japan', 'Pakistan', 'North Korea', 'South Korea', 'India',
    'Taiwan', 'NetherLands Holland Dutch', 'SPAIN Spanish',
    'Sweden Swedish', 'Mexic', 'IRAN', 'ISRAEL', 'Saudi',
    'Syria', 'Finland FINNISH', 'IRELAND IRISH', 'AUSTRIA',
    'NORWAY Norwegian', 'Switzerland swiss', 'ITALY ITALIAN',
    'MALAYSIA', 'EGYPT', 'TURKEY TURKISH', 'portugal portuguese',
    'Palestin West Bank GAZA', 'UAE United Arab Emirates emarat'
]

countries = [
    'US','GB','CA','AU','UA','RU','FR','DE','BR','CN','JP','PK',
    'KP','KR','IN','TW','NL','ES','SE','MX','IR','IL','SA','SY',
    'FI','IE','AT','NO','CH','IT','MY','EG','TR','PT','PS','AE'
]

def main():
    # Setup Bluesky client
    client = setup_bluesky_client()
    if not client:
        print("Failed to authenticate with Bluesky. Please check your credentials.")
        print("You can create an app password at: https://bsky.app/settings/app-passwords")
        return

    # Create header
    header = ['Date']
    for country in countries:
        header.append('WAR/CONFLICT ' + country)

    # Write header to file
    with open('ACA_bluesky.csv', 'w', newline='', encoding='utf-8') as f:
        writer_object = writer(f)
        writer_object.writerow(header)
        f.flush()

    # Process each time period - focusing on recent years when Bluesky was active
    for year in range(2023, 2025):  # Bluesky started in 2023
        for month in months:
            # Skip future months
            current_date = datetime.now()
            if year > current_date.year or (year == current_date.year and int(month) > current_date.month):
                continue

            conflict_row = [f"{month}/{year}"]
            
            c_index = -1
            for country in countries:
                c_index += 1
                
                country_terms = countries_s[c_index]
                print(f"Searching for {country} ({country_terms}) - {month}/{year}")
                
                try:
                    count = alternative_count_approach(client, country_terms, year, month)
                    conflict_row.append(count)
                    
                    print(f"Results for {country} {month}/{year}: {count}")
                    
                except Exception as e:
                    print(f"Error processing {country} for {month}/{year}: {e}")
                    conflict_row.append(0)
                
                # Rate limiting between country searches
                time.sleep(2)

            # Write the row for this month
            with open('ACA_bluesky.csv', 'a', newline='', encoding='utf-8') as f:
                writer_object = writer(f)
                writer_object.writerow(conflict_row)
                f.flush()
            
            print(f"Completed {month}/{year}")
            print("-" * 50)
            time.sleep(10)  # Rate limiting between months

    print("Data collection complete!")

# Simple test function to verify the API works
def test_bluesky_search():
    """Test function to verify Bluesky API is working"""
    client = setup_bluesky_client()
    if client:
        print("Testing Bluesky search...")
        count, posts = search_bluesky_posts(client, "Ukraine", limit=10)
        print(f"Test search found {count} posts")
        if posts:
            print("First post sample:")
            if hasattr(posts[0], 'post') and hasattr(posts[0].post, 'record'):
                print(posts[0].post.record.text[:100])
            elif hasattr(posts[0], 'record'):
                print(posts[0].record.text[:100])
    else:
        print("Cannot test - authentication failed")

if __name__ == "__main__":
    # Test first
    test_bluesky_search()
    
    # Then run main collection
    main()

1. Place the BlueSky username and password in the script before running

In [ ]:
"""This script analyses CVE descriptions to count occurrences of various cyber attack types and techniques over time."""

import json
import os
import re
from collections import defaultdict, Counter
from datetime import datetime
import glob

# Define the search terms with their categories
attack_categories = [
    # Initial Access
    ('Phishing', ['Phishing', 'Spear Phishing', 'Whaling', 'Business Email Compromise', 'BEC', 'Email spoofing']),
    ('Social Engineering', ['Social engineering', 'Pretexting', 'Baiting', 'Quid pro quo', 'Vishing', 'Smishing']),
    ('Email-Based Attacks', ['Email bombing', 'Email flood', 'Mailbomb', 'Spam attack', 'Malicious email attachment', 'Malicious link']),
    ('Drive-by & Watering Hole', ['Drive-by download', 'Drive-by install', 'Drive-by attack', 'Malicious redirect', 'Watering hole attack']),
    ('Exploit Kits', ['Exploit kit', 'EK', 'Angler', 'Neutrino', 'Blackhole', 'Rig', 'Sundown']),
    ('Zero-day', ['Zero-day', '0-day', 'Zero day exploit']),
    
    # Malware
    ('Malware', ['Malware', 'Malicious software', 'Malicious code', 'Malicious payload']),
    ('Ransomware', ['Ransomware', 'Crypto-ransomware', 'Locker ransomware', 'Double extortion', 'Doxware']),
    ('RaaS', ['Ransomware as a Service', 'RaaS', 'RaaS affiliate']),
    ('Ransomware Variants', ['WannaCry', 'WannaCrypt', 'Petya', 'NotPetya', 'Cryptolocker', 'Locky', 'REvil', 'Sodinokibi', 'Maze', 'LockBit', 'DarkSide', 'BlackMatter', 'Conti', 'BlackCat', 'ALPHV']),
    ('Trojans', ['Trojan', 'Trojan horse', 'Banking trojan', 'Dropper', 'Downloader']),
    ('Backdoors', ['Backdoor', 'Remote Access Trojan', 'RAT', 'Web shell']),
    ('Info-Stealers', ['Info-stealer', 'Infostealer', 'Information stealer', 'Spyware', 'Keylogger', 'Keystroke logger']),
    ('Spyware Variants', ['Pegasus', 'Predator', 'FlexiSpy', 'Redline Stealer', 'Vidar', 'Racoon Stealer']),
    ('Worms & Viruses', ['Computer worm', 'Worm', 'Self-propagating malware', 'Network worm', 'Computer virus', 'Virus']),
    ('Botnets', ['Botnet', 'Bot herder', 'C2 server', 'Command and control', 'Zombie army', 'DDoS bot']),
    ('Rootkits', ['Rootkit', 'Bootkit', 'Kernel rootkit', 'Hypervisor rootkit']),
    ('Adware', ['Adware', 'Malvertising', 'Malicious advertising', 'PUPs', 'Potentially Unwanted Program']),
    ('Wiper Malware', ['Wiper malware', 'Wiper', 'Data destruction malware', 'HermeticWiper', 'IsaacWiper', 'Shamoon']),
    ('MaaS', ['Malware as a Service', 'MaaS', 'Crimeware as a Service', 'CaaS']),
    
    # Network & Application Attacks
    ('DoS', ['Denial-of-Service', 'DoS', 'DDoS', 'Distributed Denial-of-Service', 'HTTP flood', 'SYN flood', 'Amplification attack']),
    ('Web App Attacks', ['SQL Injection', 'SQLi', 'SQLI', 'Cross-site scripting', 'XSS', 'Stored XSS', 'Reflected XSS']),
    ('Code Execution', ['Remote Code Execution', 'RCE', 'Local File Inclusion', 'LFI', 'Remote File Inclusion', 'RFI']),
    ('MITM', ['Man-in-the-middle', 'MITM', 'MitM', 'Session hijacking', 'Cookie hijacking', 'Cookie stealing']),
    ('DNS Attacks', ['DNS spoofing', 'DNS cache poisoning', 'DNS hijacking', 'DNS tunneling', 'Pharming']),
    ('Password Attacks', ['Password attack', 'Credential theft', 'Credential stuffing', 'Brute force', 'Dictionary attack', 'Password spraying', 'Pass the hash']),
    
    # Monetization & Fraud
    ('Carding', ['Carding', 'Card fraud', 'Credit card theft', 'Payment card fraud', 'Fullz', 'Fulls', 'Card dump', 'Dump', 'BIN', 'CVV', 'CVV2']),
    ('Account Takeover', ['Bank log', 'Account takeover', 'ATO', 'eWhoring']),
    ('Cryptojacking', ['Cryptojacking', 'Illegal cryptocurrency mining', 'Coin miner malware', 'Crypto-miner']),
    ('Data Breaches', ['Data breach', 'Data leak', 'Data exfiltration', 'Data theft', 'Database dump', 'Leaked database']),
    ('Underground Markets', ['Dark web market', 'Underground market', 'Darknet market', 'Carding forum', 'Hacking forum']),
    ('Doxing', ['Doxing', 'Doxxing', 'Swatting']),
    
    # Advanced Threats
    ('Lateral Movement', ['Lateral movement', 'Privilege escalation', 'Persistence', 'Defense evasion', 'Living off the land', 'LotL']),
    ('APTs', ['Advanced Persistent Threat', 'APT', 'Nation state attack', 'State-sponsored']),
    ('Supply Chain', ['Supply chain attack', 'Third-party attack', 'Software supply chain', 'Dependency confusion']),
    ('Insider Threats', ['Insider threat', 'Malicious insider', 'Rogue employee', 'Insider fraud']),
    ('IoT Attacks', ['IoT attack', 'IoT botnet', 'Mirai', 'Operational Technology', 'OT attack', 'ICS attack']),
    ('AI Security', ['Adversarial attack', 'AI poisoning', 'Model inversion', 'Data poisoning', 'Training data poisoning']),
]

def extract_text_from_json(json_data):
    """Extract all text content from JSON data for searching"""
    text_parts = []
    
    def extract_from_dict(d):
        for key, value in d.items():
            if isinstance(value, str):
                text_parts.append(value.lower())
            elif isinstance(value, dict):
                extract_from_dict(value)
            elif isinstance(value, list):
                extract_from_list(value)
    
    def extract_from_list(lst):
        for item in lst:
            if isinstance(item, str):
                text_parts.append(item.lower())
            elif isinstance(item, dict):
                extract_from_dict(item)
            elif isinstance(item, list):
                extract_from_list(item)
    
    if isinstance(json_data, dict):
        extract_from_dict(json_data)
    elif isinstance(json_data, list):
        extract_from_list(json_data)
    
    return " ".join(text_parts)

def search_terms_in_text(text, terms):
    """Search for terms in text (case-insensitive)"""
    found_terms = []
    text_lower = text.lower()
    
    for term in terms:
        # Use word boundaries to avoid partial matches
        pattern = r'\b' + re.escape(term.lower()) + r'\b'
        if re.search(pattern, text_lower):
            found_terms.append(term)
    
    return found_terms

def parse_date(date_string):
    """Parse date from datePublic field and make timezone-naive"""
    try:
        if date_string:
            # Handle different date formats
            if 'T' in date_string:
                dt = datetime.fromisoformat(date_string.replace('Z', '+00:00'))
                # Convert to naive datetime by removing timezone info
                return dt.replace(tzinfo=None)
            else:
                return datetime.strptime(date_string, '%Y-%m-%d')
    except (ValueError, AttributeError):
        return None
    return None

def process_json_files(directory_path):
    """Process all JSON files in the directory"""
    monthly_counts = defaultdict(lambda: defaultdict(int))
    all_mentions = defaultdict(set)  # Store unique mentions per category
    file_pattern = os.path.join(directory_path, "**", "*.json")
    files_processed = 0
    
    # Create timezone-naive datetime objects for comparison
    start_date = datetime(2011, 7, 1)
    end_date = datetime(2024, 12, 31)
    
    for file_path in glob.glob(file_pattern, recursive=True):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract datePublic
            date_public = None
            try:
                date_public = data.get('containers', {}).get('cna', {}).get('datePublic')
                if not date_public:
                    date_public = data.get('cveMetadata', {}).get('datePublished')
            except (AttributeError, KeyError):
                pass
            
            date_obj = parse_date(date_public)
            if not date_obj:
                continue
            
            # Skip if outside date range (all are timezone-naive now)
            if date_obj < start_date or date_obj > end_date:
                continue
            
            month_key = date_obj.strftime('%Y-%m')
            
            # Extract all text content
            all_text = extract_text_from_json(data)
            
            # Search for each category
            for category, terms in attack_categories:
                found_terms = search_terms_in_text(all_text, terms)
                if found_terms:
                    monthly_counts[month_key][category] += 1
                    all_mentions[category].update(found_terms)
            
            files_processed += 1
            if files_processed % 1000 == 0:
                print(f"Processed {files_processed} files...")
                
        except (json.JSONDecodeError, KeyError, UnicodeDecodeError) as e:
            print(f"Error processing {file_path}: {e}")
            continue
    
    print(f"Total files processed: {files_processed}")
    return monthly_counts, all_mentions

def generate_output(monthly_counts, all_mentions, output_dir="output"):
    """Generate output files with results"""
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Monthly counts CSV
    with open(os.path.join(output_dir, "monthly_counts.csv"), 'w', encoding='utf-8') as f:
        # Write header
        categories = [cat for cat, _ in attack_categories]
        header = "Month," + ",".join(categories) + ",Total\n"
        f.write(header)
        
        # Generate all months from July 2011 to December 2024
        all_months = []
        current_date = datetime(2011, 7, 1)
        end_date = datetime(2024, 12, 31)
        
        while current_date <= end_date:
            all_months.append(current_date.strftime('%Y-%m'))
            if current_date.month == 12:
                current_date = current_date.replace(year=current_date.year + 1, month=1)
            else:
                current_date = current_date.replace(month=current_date.month + 1)
        
        # Write data for each month
        for month in all_months:
            row_data = [str(monthly_counts[month].get(category, 0)) for category in categories]
            total = sum(monthly_counts[month].values())
            f.write(f"{month},{','.join(row_data)},{total}\n")
    
    # 2. Unique mentions per category
    with open(os.path.join(output_dir, "unique_mentions.txt"), 'w', encoding='utf-8') as f:
        f.write("UNIQUE MENTIONS BY CATEGORY\n")
        f.write("=" * 50 + "\n\n")
        
        for category, terms in attack_categories:
            found_terms = sorted(all_mentions[category])
            f.write(f"{category}:\n")
            if found_terms:
                for term in found_terms:
                    f.write(f"  - {term}\n")
            else:
                f.write("  (No mentions found)\n")
            f.write("\n")
    
    # 3. Summary statistics
    with open(os.path.join(output_dir, "summary_stats.txt"), 'w', encoding='utf-8') as f:
        f.write("SUMMARY STATISTICS\n")
        f.write("=" * 30 + "\n\n")
        
        total_mentions = sum(len(mentions) for mentions in all_mentions.values())
        categories_with_mentions = sum(1 for mentions in all_mentions.values() if mentions)
        
        f.write(f"Total categories with mentions: {categories_with_mentions}/{len(attack_categories)}\n")
        f.write(f"Total unique term mentions: {total_mentions}\n")
        f.write(f"Time period: July 2011 - December 2024\n\n")
        
        # Top categories by frequency
        category_totals = []
        for category, _ in attack_categories:
            total = sum(monthly_counts[month].get(category, 0) for month in monthly_counts)
            category_totals.append((category, total))
        
        category_totals.sort(key=lambda x: x[1], reverse=True)
        
        f.write("TOP 10 CATEGORIES BY MENTION FREQUENCY:\n")
        for i, (category, total) in enumerate(category_totals[:10], 1):
            f.write(f"{i:2d}. {category}: {total} mentions\n")

def main():
    # Configure your directory path here
    data_directory = "C:/Users/Bella/Documents/GitHub/cvelistV5/cves"  # Change this to your directory path
    
    if not os.path.exists(data_directory):
        print(f"Error: Directory '{data_directory}' does not exist.")
        return
    
    print("Starting analysis...")
    print("This may take a while depending on the number of files...")
    
    monthly_counts, all_mentions = process_json_files(data_directory)
    generate_output(monthly_counts, all_mentions)
    
    print("Analysis complete!")
    print("Output files generated in 'output' directory:")
    print("  - monthly_counts.csv")
    print("  - unique_mentions.txt") 
    print("  - summary_stats.txt")

if __name__ == "__main__":
    main()

1. Clone the Git Repository: https://github.com/CVEProject/cvelistV5
2. Input the location of the cloned repository to the "data_directory" field
3. Run the script